# Решение v12: поиск 50 кандидатов для запроса по услугам

Кандидат на финальную отправку. Локальный matched Recall@50: **0.962225** против **0.961181** у отправленного v11.

Notebook полностью воспроизводит `experiments/results/v12/answer.csv` из локальных данных и сохранённых моделей.


## Перед запуском

Положите `train.parquet`, `benchmark_queries.parquet` и
`benchmark_items.parquet` рядом с notebook. Установите `requirements.txt`.
Выполните **Restart Kernel → Run All**: notebook создаст `answer.csv` в той
же папке и проверит его SHA-256. Веса и готовые векторы включены в архив;
внешний API не требуется. CPU поддерживается, GPU ускоряет расчёт.

`train` содержит наблюдаемые пары «запрос — выбранное объявление».
`benchmark_queries` содержит запросы без меток, `benchmark_items` — корпус
для поиска. Идентификаторы всё время остаются строками. Метки benchmark
нигде не используются.

In [1]:
from pathlib import Path
import gc, hashlib, html, json, re, sys, unicodedata
from types import MappingProxyType
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'train.parquet').exists()), Path.cwd())
if not (ROOT / 'train.parquet').exists():
    raise FileNotFoundError('Запустите notebook из папки с тремя Parquet')
if (ROOT / '.inspection_deps').is_dir():
    sys.path.insert(0, str(ROOT / '.inspection_deps'))
import numpy as np
import pandas as pd
SEED = 260926
CONFIG = {'validation_texts': 1400, 'cold_item_fraction': .9, 'development_fraction': .5}
RANKER_CONFIG = {'audit_queries': 600, 'fresh_audit_queries': 600, 'training_queries': 4000}
QUERY_COLS = ['search_query', 'search_location_id', 'search_is_delivery_search',
              'search_infm_params_text', 'search_category']
CACHE = ROOT / 'artifacts' / 'validation-v5'
CACHE.mkdir(parents=True, exist_ok=True)
train = pd.read_parquet(ROOT / 'train.parquet', columns=QUERY_COLS +
                       ['item_id', 'item_location_id', 'item_microcat_id'])
queries = pd.read_parquet(ROOT / 'benchmark_queries.parquet')
items = pd.read_parquet(ROOT / 'benchmark_items.parquet', columns=['item_id'])
items = items.sort_values('item_id').reset_index(drop=True)
ITEM_IDS = items.item_id.to_numpy(dtype=str)
ITEM_TO_ROW = {value: row for row, value in enumerate(ITEM_IDS)}
assert items.item_id.is_unique and queries.query_id.is_unique
for frame in [train, queries]:
    for column in ['search_query', 'search_infm_params_text']:
        frame[column] = frame[column].fillna('').astype('string[pyarrow]')


In [2]:
def normalize_text(value):
    value = unicodedata.normalize("NFKC", html.unescape(str(value))).lower().replace("ё", "е")
    value = re.sub(r"<[^>]+>", " ", value)
    return re.sub(r"\s+", " ", value).strip()

for frame in [train, queries]:
    frame['query_norm'] = frame.search_query.map(normalize_text).astype('string[pyarrow]')
    frame['context_key'] = [hashlib.sha256(json.dumps([str(v) for v in row],
        ensure_ascii=False).encode()).hexdigest()[:24]
        for row in frame[QUERY_COLS].itertuples(index=False, name=None)]
def select_query_contexts(history, count, seed):
    eligible_pairs = history[history.item_id.isin(ITEM_TO_ROW)]
    local_rng = np.random.default_rng(seed)
    texts = np.sort(eligible_pairs.query_norm.unique().astype(str))
    chosen_texts = local_rng.choice(texts, min(count, len(texts)), replace=False)
    contexts = eligible_pairs[eligible_pairs.query_norm.isin(chosen_texts)].drop_duplicates("context_key")
    chosen = []
    for _, group in contexts.sort_values("context_key").groupby("query_norm", sort=True):
        chosen.append(group.iloc[int(local_rng.integers(len(group)))])
    return pd.DataFrame(chosen)[[*QUERY_COLS, "query_norm", "context_key"]].sort_values("context_key").reset_index(drop=True)

def query_labels(history, query_frame):
    pairs = history[history.context_key.isin(query_frame.context_key) & history.item_id.isin(ITEM_TO_ROW)]
    grouped = pairs.groupby("context_key").item_id.agg(lambda s: set(ITEM_TO_ROW[x] for x in s))
    return [grouped.get(key, set()) for key in query_frame.context_key]

def purge_history(history, query_frame, relevant, seed):
    positives = np.array(sorted({ITEM_IDS[i] for truth in relevant for i in truth}))
    local_rng = np.random.default_rng(seed)
    cold_ids = set(local_rng.choice(positives, int(CONFIG["cold_item_fraction"] * len(positives)), replace=False))
    clean = history[~history.query_norm.isin(query_frame.query_norm) & ~history.item_id.isin(cold_ids)].copy()
    assert not set(query_frame.query_norm) & set(clean.query_norm)
    assert not cold_ids & set(clean.item_id)
    return clean, cold_ids


## 1. Валидация без утечки меток

Один и тот же короткий текст встречается в разных контекстах. Поэтому
разделение проводится по нормализованному тексту и контексту запроса, а
собственные взаимодействия контрольного запроса исключаются из истории.
Сначала строится неизменяемая таблица положительных пар, затем из доступной
истории создаются варианты с новой и знакомой формулировкой.

In [3]:
pairs = train.drop_duplicates(["context_key", "item_id"]).copy()
eligible = pairs[pairs.item_id.isin(ITEM_TO_ROW)].copy()
rng = np.random.default_rng(SEED)
text_pool = np.sort(eligible.query_norm.unique().astype(str))
selected_texts = rng.choice(text_pool, size=min(CONFIG["validation_texts"], len(text_pool)), replace=False)
selected = eligible[eligible.query_norm.isin(selected_texts)]
context_pool = selected[["query_norm", "context_key"]].drop_duplicates().sort_values(["query_norm", "context_key"])
chosen_keys = []
for _, group in context_pool.groupby("query_norm", sort=True):
    chosen_keys.append(group.context_key.iloc[int(rng.integers(len(group)))])
validation_pairs = selected[selected.context_key.isin(chosen_keys)]
validation = validation_pairs.drop_duplicates("context_key")[[*QUERY_COLS, "query_norm", "context_key"]].copy()
validation = validation.sort_values("context_key").reset_index(drop=True)
labels_by_key = validation_pairs.groupby("context_key").item_id.agg(lambda s: sorted(set(s)))
labels = [set(ITEM_TO_ROW[item_id] for item_id in labels_by_key[key]) for key in validation.context_key]
val_positive_ids = np.sort(validation_pairs.item_id.unique().astype(str))
cold_item_ids = set(rng.choice(val_positive_ids, size=int(CONFIG["cold_item_fraction"] * len(val_positive_ids)), replace=False))
fit_pairs = pairs[~pairs.query_norm.isin(selected_texts) & ~pairs.item_id.isin(cold_item_ids)].copy()
assert not set(validation.query_norm) & set(fit_pairs.query_norm)
assert not cold_item_ids & set(fit_pairs.item_id)
assert all(labels)
split_order = rng.permutation(len(validation))
dev_indices = np.sort(split_order[:int(len(validation) * CONFIG["development_fraction"])])
holdout_indices = np.sort(split_order[int(len(validation) * CONFIG["development_fraction"]):])
validation["split"] = "holdout"
validation.loc[dev_indices, "split"] = "development"
validation["relevant_item_ids"] = [" ".join(labels_by_key[key]) for key in validation.context_key]
validation.to_parquet(CACHE / "validation.parquet", index=False)
print("History pairs:", len(fit_pairs), "; validation queries:", len(validation))
print("Development:", len(dev_indices), "; holdout:", len(holdout_indices))
print("Mean positives:", round(np.mean([len(x) for x in labels]), 3))
print("Actual unseen positive item fraction:", round(1 - validation_pairs.item_id.isin(fit_pairs.item_id).mean(), 4))
# Полные тексты train больше не нужны; сохраняем компактную историю для обучения.
HISTORY_COLS = [*QUERY_COLS, "query_norm", "context_key", "item_id", "item_location_id", "item_microcat_id"]
history_all = pairs[HISTORY_COLS].copy()
history_fit = fit_pairs[HISTORY_COLS].copy()
del train, pairs, fit_pairs, eligible, selected, validation_pairs
gc.collect()


History pairs: 428634 ; validation queries: 1400
Development: 700 ; holdout: 700
Mean positives: 1.026
Actual unseen positive item fraction: 0.952


In [4]:
audit_queries = select_query_contexts(history_fit, RANKER_CONFIG["audit_queries"], SEED + 101)
audit_labels = query_labels(history_fit, audit_queries)
ranker_history, audit_cold_ids = purge_history(history_fit, audit_queries, audit_labels, SEED + 102)
original_training_queries = select_query_contexts(ranker_history, RANKER_CONFIG["training_queries"], SEED + 103)
# These 600 texts were not direct training queries in the already inspected v0.2
# model. Remove their labels from all history before fitting the new comparison.
fresh_source = ranker_history[~ranker_history.query_norm.isin(original_training_queries.query_norm)]
fresh_queries = select_query_contexts(fresh_source, RANKER_CONFIG["fresh_audit_queries"], SEED + 501)
fresh_labels = query_labels(ranker_history, fresh_queries)
ranker_history, fresh_cold_ids = purge_history(ranker_history, fresh_queries, fresh_labels, SEED + 502)
training_queries = select_query_contexts(ranker_history, RANKER_CONFIG["training_queries"], SEED + 103)
assert all(fresh_labels)
assert not set(fresh_queries.query_norm) & set(original_training_queries.query_norm)
assert not set(fresh_queries.query_norm) & set(training_queries.query_norm)
assert not set(fresh_queries.query_norm) & set(validation.query_norm)
assert not set(fresh_queries.query_norm) & set(audit_queries.query_norm)
assert not set(fresh_queries.query_norm) & set(ranker_history.query_norm)
assert not fresh_cold_ids & set(ranker_history.item_id)
fresh_protocol = {"fresh_queries": len(fresh_queries), "development_queries": len(validation) + len(audit_queries),
    "training_queries": len(training_queries), "query_text_overlap": 0, "purged_item_overlap": 0,
    "old_training_query_text_overlap": 0, "purged_positive_items": len(fresh_cold_ids),
    "used_for_model_selection": False}
fresh_protocol["role"] = "historical_v0.3_slice_now_development"
fresh_protocol["used_for_model_selection"] = True
(CACHE / "prior_split_protocol.json").write_text(json.dumps(fresh_protocol, indent=2), encoding="utf-8")
del fresh_source



old_direct_texts = set(training_queries.query_norm) | set(original_training_queries.query_norm)
old_direct_ids = {ITEM_IDS[i] for truth in query_labels(ranker_history, training_queries) for i in truth}
overlap_texts = set(ranker_history.loc[ranker_history.item_id.isin(old_direct_ids), 'query_norm'])
audit_source = ranker_history[~ranker_history.query_norm.isin(old_direct_texts | overlap_texts)]
new_audit = select_query_contexts(audit_source, 800, SEED + 702)
old_new_audit_labels = query_labels(ranker_history, new_audit)
development = pd.concat([validation[[*QUERY_COLS, 'query_norm', 'context_key']],
                         audit_queries, fresh_queries, new_audit], ignore_index=True)
assert development.context_key.is_unique


### Проверка полноты разметки и групп

Исходные положительные метки хранятся отдельно от очищенной истории: удаление
наблюдения из признаков не должно удалять его из контрольной разметки.
Этот инвариант исправил потерю семи положительных пар в ранней версии.

In [5]:
"""Validation contracts for v5; the notebook embeds these functions verbatim.

Gold labels come from the complete interaction table. History is a separate,
mutable view: removing information from it must never remove evaluation labels.
"""
from types import MappingProxyType

import numpy as np
import pandas as pd


def freeze_gold(full_pairs, item_to_row):
    """Snapshot all observed positives in the searchable corpus, once.

    Both the mapping and its values are immutable. Corpus membership is the only
    label filter; geography and historical purges do not affect the gold set.
    """
    eligible = full_pairs.loc[full_pairs.item_id.isin(item_to_row),
                              ['context_key', 'item_id']].drop_duplicates()
    grouped = eligible.groupby('context_key', sort=True).item_id.agg(
        lambda values: frozenset(item_to_row[value] for value in values))
    return MappingProxyType(grouped.to_dict())


def labels_from_gold(gold, query_frame):
    """Return immutable labels in query order; never accept a history argument."""
    return [gold.get(key, frozenset()) for key in query_frame.context_key]


def history_for_queries(history, query_frame, truths, item_ids, mode,
                        cold_fraction, seed, excluded_contexts=(),
                        excluded_texts=(), excluded_items=()):
    """Build an isolated feature history for a specified evaluation regime.

    unseen_text removes every context of the held text. held_context removes
    own context interactions but permits other contexts of the same text.
    Without timestamps, this does not evaluate repeats of the identical context.
    """
    if mode not in {'unseen_text', 'held_context'}:
        raise ValueError(f'Unknown history mode: {mode}')
    if not 0 <= cold_fraction <= 1:
        raise ValueError('cold_fraction must be in [0, 1]')
    if len(query_frame) != len(truths):
        raise ValueError('One gold set is required per query')
    positive_ids = np.array(sorted({item_ids[i] for truth in truths for i in truth}),
                            dtype=str)
    random = np.random.default_rng(seed)
    cold = set(random.choice(positive_ids, int(cold_fraction * len(positive_ids)),
                             replace=False))
    blocked_contexts = set(query_frame.context_key) | set(excluded_contexts)
    blocked_texts = set(excluded_texts)
    if mode == 'unseen_text':
        blocked_texts.update(query_frame.query_norm)
    blocked_items = cold | set(excluded_items)
    fit = history.loc[~history.context_key.isin(blocked_contexts)
                      & ~history.query_norm.isin(blocked_texts)
                      & ~history.item_id.isin(blocked_items)].copy()
    assert not blocked_contexts & set(fit.context_key)
    assert not blocked_items & set(fit.item_id)
    assert not blocked_texts & set(fit.query_norm)
    known = query_frame.query_norm.isin(fit.query_norm).to_numpy(dtype=bool)
    if mode == 'unseen_text':
        assert not known.any()
    return fit, cold, known


def oof_assignments(query_frame, mode, folds=3, seed=260926):
    """Group by text for cold queries, by context for held-context queries.

    A held context includes all its positives and duplicate rows. Different
    contexts of one text may appear in the fit history in the second regime.
    """
    if mode not in {'unseen_text', 'held_context'}:
        raise ValueError(f'Unknown history mode: {mode}')
    if folds < 2:
        raise ValueError('At least two folds are required')
    column = 'query_norm' if mode == 'unseen_text' else 'context_key'
    values = np.sort(query_frame[column].unique().astype(str))
    random = np.random.default_rng(seed)
    shuffled = values[random.permutation(len(values))]
    mapping = {key: position % folds for position, key in enumerate(shuffled)}
    return query_frame[column].map(mapping).to_numpy(dtype=np.int32)


def history_coverage(query_frame, truths, fit, item_ids):
    """Measure actual availability rather than infer it from a regime's name."""
    known = query_frame.query_norm.isin(fit.query_norm).to_numpy(dtype=bool)
    observed_items = set(fit.item_id)
    fractions = [sum(item_ids[i] in observed_items for i in truth) / len(truth)
                 for truth in truths if truth]
    return {'contexts': len(query_frame), 'positive_pairs': sum(map(len, truths)),
            'known_text_fraction': float(known.mean()) if len(known) else None,
            'positive_item_seen_macro_fraction': float(np.mean(fractions)) if fractions else None,
            'own_context_overlap': len(set(query_frame.context_key) & set(fit.context_key))}


gold = freeze_gold(history_all, ITEM_TO_ROW)
development_truths = labels_from_gold(gold, development)
assert all(development_truths)
restored_new = labels_from_gold(gold, new_audit)
label_changes = {'old_new_audit_positives': sum(map(len, old_new_audit_labels)),
    'full_new_audit_positives': sum(map(len, restored_new)),
    'affected_new_audit_contexts': sum(a != b for a, b in zip(old_new_audit_labels, restored_new))}
assert label_changes == {'old_new_audit_positives': 815, 'full_new_audit_positives': 822,
                         'affected_new_audit_contexts': 6}
known_text = queries.query_norm.isin(history_all.query_norm)
known_context = queries.context_key.isin(history_all.context_key)
target = {'unseen_text_fraction': float((~known_text).mean()),
    'known_text_new_context_fraction': float((known_text & ~known_context).mean()),
    'exact_context_fraction': float(known_context.mean()),
    'exact_context_evaluation_limitation': 'No timestamps or independent repeated events'}
assert np.isclose(sum(target[k] for k in ['unseen_text_fraction',
    'known_text_new_context_fraction', 'exact_context_fraction']), 1)
print('Исправление меток:', json.dumps(label_changes, ensure_ascii=False))
print('Режимы бенчмарка:', json.dumps(target, ensure_ascii=False))
profiles = []
for mode in ['unseen_text', 'held_context']:
    for fraction in [0., .5, .9, 1.]:
        fit, cold, _ = history_for_queries(history_all, development,
            development_truths, ITEM_IDS, mode, fraction, SEED + 801)
        profile = history_coverage(development, development_truths, fit, ITEM_IDS)
        profile.update(mode=mode, cold_fraction=fraction, cold_items=len(cold))
        profiles.append(profile)
        assert profile['own_context_overlap'] == 0
        del fit
print(pd.DataFrame(profiles).to_string(index=False))


Исправление меток: {"old_new_audit_positives": 815, "full_new_audit_positives": 822, "affected_new_audit_contexts": 6}
Режимы бенчмарка: {"unseen_text_fraction": 0.6264274061990212, "known_text_new_context_fraction": 0.3295269168026101, "exact_context_fraction": 0.04404567699836868, "exact_context_evaluation_limitation": "No timestamps or independent repeated events"}
 contexts  positive_pairs  known_text_fraction  positive_item_seen_macro_fraction  own_context_overlap         mode  cold_fraction  cold_items
     3400            3480             0.000000                           0.456000                    0  unseen_text            0.0           0
     3400            3480             0.000000                           0.232951                    0  unseen_text            0.5        1683
     3400            3480             0.000000                           0.045539                    0  unseen_text            0.9        3029
     3400            3480             0.000000           

### Контексты обучения и контрольные режимы

Отдельно проверяются новые тексты запросов и известные тексты в новом
контексте. Локальная метрика считается на полном пуле кандидатов, а не на
подвыборке отрицательных примеров, которая использовалась для обучения.

In [6]:
def context_sample(history, count, seed, include=None):
    eligible = history[history.item_id.isin(ITEM_TO_ROW)].drop_duplicates('context_key')
    local = np.random.default_rng(seed)
    eligible = eligible.sort_values('context_key').iloc[local.permutation(len(eligible))]
    bounded = eligible.groupby('query_norm', sort=False).head(EXPERIMENT_CONFIG['contexts_per_text'])
    if include is not None:
        bounded = pd.concat([include, bounded], ignore_index=True).drop_duplicates('context_key')
    return bounded.head(count)[[*QUERY_COLS,'query_norm','context_key']].sort_values('context_key').reset_index(drop=True)

# Reserve text groups not used for direct supervised fitting in v4. Reconstruct
# that list locally rather than depend on an untracked cached parquet file.
EXPERIMENT_CONFIG = {'contexts_per_text': 4}
old_clean, _ = purge_history(ranker_history, new_audit, old_new_audit_labels, SEED + 703)
old_expanded = context_sample(old_clean, 20000, SEED + 704, training_queries)
previous_direct_texts = set(old_expanded.query_norm) | set(development.query_norm)
control_source = history_all[~history_all.query_norm.isin(previous_direct_texts)]
held_control = select_query_contexts(control_source, 600, SEED + 805)
held_control_truths = labels_from_gold(gold, held_control)
assert len(held_control) == 600 and all(held_control_truths)
assert not set(held_control.query_norm) & previous_direct_texts
evaluation_contexts = pd.concat([development, held_control], ignore_index=True)
evaluation_truths = development_truths + held_control_truths
training_history, development_cold, _ = history_for_queries(history_all,
    evaluation_contexts, evaluation_truths, ITEM_IDS, 'unseen_text', .9, SEED + 802)
candidate_contexts = history_all[history_all.context_key.isin(gold)].drop_duplicates('context_key')
candidate_contexts = candidate_contexts[~candidate_contexts.query_norm.isin(evaluation_contexts.query_norm)]
candidate_contexts = candidate_contexts[[*QUERY_COLS, 'query_norm', 'context_key']].sort_values('context_key').reset_index(drop=True)
candidate_truths = labels_from_gold(gold, candidate_contexts)
assert all(candidate_truths)
assert not set(candidate_contexts.query_norm) & set(evaluation_contexts.query_norm)
fold_profiles, assignment_rows = [], []
for mode in ['unseen_text', 'held_context']:
    assignments = oof_assignments(candidate_contexts, mode, 3, SEED + 803)
    for fold in range(3):
        positions = np.flatnonzero(assignments == fold)
        q = candidate_contexts.iloc[positions]
        truth = [candidate_truths[i] for i in positions]
        fit, cold, actual_known = history_for_queries(training_history, q, truth,
            ITEM_IDS, mode, .9, SEED + 804 + fold)
        profile = history_coverage(q, truth, fit, ITEM_IDS)
        profile.update(mode=mode, fold=fold, cold_items=len(cold), history_pairs=len(fit))
        fold_profiles.append(profile)
        assignment_rows.extend({'context_key': key, 'mode': mode, 'fold': fold,
            'actual_known_text': bool(known)} for key, known in zip(q.context_key, actual_known))
        assert profile['own_context_overlap'] == 0
        if mode == 'unseen_text':
            assert profile['known_text_fraction'] == 0
        del fit
        gc.collect()
prepared = candidate_contexts.copy()
prepared['relevant_item_ids'] = [' '.join(ITEM_IDS[i] for i in sorted(truth)) for truth in candidate_truths]
prepared.to_parquet(CACHE / 'training_contexts.parquet', index=False)
prepared_dev = development.copy()
prepared_dev['relevant_item_ids'] = [' '.join(ITEM_IDS[i] for i in sorted(truth)) for truth in development_truths]
prepared_dev.to_parquet(CACHE / 'development_contexts.parquet', index=False)
prepared_control = held_control.copy()
prepared_control['relevant_item_ids'] = [' '.join(ITEM_IDS[i] for i in sorted(truth)) for truth in held_control_truths]
prepared_control.to_parquet(CACHE / 'held_control_contexts.parquet', index=False)
pd.DataFrame(assignment_rows).to_parquet(CACHE / 'oof_assignments.parquet', index=False)
pd.DataFrame(profiles).to_csv(CACHE / 'history_regimes.csv', index=False)
pd.DataFrame(fold_profiles).to_csv(CACHE / 'oof_history_audit.csv', index=False)
report = {'stage': 'validation-v5', 'gold_source': 'complete unpurged train, corpus positives only',
    'gold_contexts': len(gold), 'gold_positive_pairs': sum(map(len, gold.values())),
    'label_correction': label_changes, 'benchmark_regimes': target,
    'development_contexts': len(development), 'development_role': 'previously viewed development',
    'held_control_contexts': len(held_control), 'held_control_direct_text_overlap': 0,
    'held_control_limitation': 'Frozen v4 priors may have used these interactions in auxiliary history; strict validation requires refitting affected priors',
    'training_contexts': len(candidate_contexts),
    'training_texts': int(candidate_contexts.query_norm.nunique()),
    'training_positives': sum(map(len, candidate_truths)),
    'oof_audits': fold_profiles, 'evaluation_history_profiles': profiles,
    'new_models_fitted': False, 'answer_csv_modified': False,
    'next_stage': 'full-pool retrieval, stronger hard negatives, mixed-regime ranker training'}
(CACHE / 'protocol.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({k: v for k, v in report.items() if k not in ['oof_audits', 'evaluation_history_profiles']},
                 ensure_ascii=False, indent=2))
print(pd.DataFrame(fold_profiles).to_string(index=False))


{
  "stage": "validation-v5",
  "gold_source": "complete unpurged train, corpus positives only",
  "gold_contexts": 26556,
  "gold_positive_pairs": 29572,
  "label_correction": {
    "old_new_audit_positives": 815,
    "full_new_audit_positives": 822,
    "affected_new_audit_contexts": 6
  },
  "benchmark_regimes": {
    "unseen_text_fraction": 0.6264274061990212,
    "known_text_new_context_fraction": 0.3295269168026101,
    "exact_context_fraction": 0.04404567699836868,
    "exact_context_evaluation_limitation": "No timestamps or independent repeated events"
  },
  "development_contexts": 3400,
  "development_role": "previously viewed development",
  "held_control_contexts": 600,
  "held_control_direct_text_overlap": 0,
  "held_control_limitation": "Frozen v4 priors may have used these interactions in auxiliary history; strict validation requires refitting affected priors",
  "training_contexts": 19031,
  "training_texts": 8174,
  "training_positives": 21480,
  "new_models_fitted": f

## 2. Подготовка корпуса и быстрый поиск

Следующие ячейки задают рабочий индекс. Начальный аудит выше намеренно
загружает те же Parquet-файлы отдельно: он проверяет состав данных и
валидационные ограничения до построения поисковых структур. Повторная
инициализация ниже создаёт каноническое состояние для применения модели.

In [7]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'train.parquet').exists()), Path.cwd())
if not (ROOT / "train.parquet").exists() and (ROOT / "AvitoTest/train.parquet").exists():
    ROOT = ROOT / "AvitoTest"
if (ROOT / ".inspection_deps").is_dir():
    sys.path.insert(0, str(ROOT / ".inspection_deps"))

import gc
import hashlib
import html
import json
import os
import re
import time
import unicodedata
from collections import Counter, defaultdict
from functools import lru_cache
import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import sklearn
import snowballstemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier
from scipy.stats import rankdata
from threadpoolctl import threadpool_limits

SEED = 260926
CONFIG = {
    "seed": SEED, "validation_texts": 1400, "development_fraction": 0.5,
    "cold_item_fraction": 0.9, "description_chars": 4000,
    "params_chars": 1800, "word_features": 240_000,
    "char_features": 220_000, "retrieval_pool": 500,
    "version": "semantic-v1",
}
RANKER_CONFIG = {
    "training_queries": 4000, "audit_queries": 600, "fresh_audit_queries": 600, "folds": 3,
    "hard_negatives": 100, "random_negatives": 60,
    "iterations": [120, 240], "leaf_options": [15, 31],
    "learning_rate": 0.08, "l2_regularization": 10.0,
    "blend_options": [0.5, 0.75, 1.0], "feature_version": 1,
}
SEMANTIC_CONFIG = {
    "model_id": "intfloat/multilingual-e5-small",
    "revision": "614241f622f53c4eeff9890bdc4f31cfecc418b3",
    "max_length": 192, "description_chars": 1200, "params_chars": 256, "batch_size": 128,
    "retrieval_pool": 500, "temperature": 0.05, "dtype": "float32",
}
CACHE = ROOT / "artifacts" / CONFIG["version"]
CACHE.mkdir(parents=True, exist_ok=True)
USE_CACHE = os.environ.get("AVITO_REBUILD_CACHE", "0") != "1"
threadpool_limits(limits=8)
np.random.seed(SEED)
print("Project:", ROOT)
print("Versions:", {"python": sys.version.split()[0], "pandas": pd.__version__,
                    "numpy": np.__version__, "sklearn": sklearn.__version__})


Project: D:\JupyterProjects\AvitoTest
Versions: {'python': '3.14.6', 'pandas': '3.0.6', 'numpy': '2.5.3', 'sklearn': '1.9.1'}


In [8]:
QUERY_COLS = ["search_query", "search_location_id", "search_is_delivery_search",
              "search_infm_params_text", "search_category"]
train = pd.read_parquet(ROOT / "train.parquet")
queries = pd.read_parquet(ROOT / "benchmark_queries.parquet")
items = pd.read_parquet(ROOT / "benchmark_items.parquet").sort_values("item_id").reset_index(drop=True)
for frame in [train, queries, items]:
    for column in frame.columns:
        if column.endswith("_raw") or column.endswith("_text") or column == "search_query":
            frame[column] = frame[column].fillna("").astype("string[pyarrow]")
    for column in ["item_price", "item_latitude", "item_longitude"]:
        if column in frame:
            frame[column] = pd.to_numeric(frame[column], errors="coerce").astype("float64")

assert len(queries) == queries.query_id.nunique()
assert queries.query_id.str.len().eq(16).all()
assert items.item_id.is_unique
assert items.item_id.str.fullmatch(r"[0-9a-f]{16}").all()
assert train.item_id.str.fullmatch(r"[0-9a-f]{16}").all()
assert set(QUERY_COLS).issubset(train.columns) and set(QUERY_COLS).issubset(queries.columns)
ITEM_IDS = items.item_id.to_numpy(dtype=str)
ITEM_TO_ROW = {item_id: index for index, item_id in enumerate(ITEM_IDS)}
print(pd.DataFrame([
    {"file": name, "rows": len(frame), "columns": frame.shape[1],
     "memory_MiB": round(frame.memory_usage(deep=True).sum() / 2**20, 1)}
    for name, frame in [("train", train), ("queries", queries), ("items", items)]
]).to_string(index=False))


   file   rows  columns  memory_MiB
  train 497673       19      2197.0
queries   2452        6         0.3
  items 189212       14       801.8


In [9]:
def normalize_text(value):
    value = unicodedata.normalize("NFKC", html.unescape(str(value))).lower().replace("ё", "е")
    value = re.sub(r"<[^>]+>", " ", value)
    return re.sub(r"\s+", " ", value).strip()

for frame in [train, queries]:
    frame["query_norm"] = frame.search_query.map(normalize_text).astype("string[pyarrow]")
    # JSON сохраняет границы признаков: конкатенация без разделителей небезопасна.
    frame["context_key"] = [
        hashlib.sha256(json.dumps([str(v) for v in row], ensure_ascii=False).encode()).hexdigest()[:24]
        for row in frame[QUERY_COLS].itertuples(index=False, name=None)
    ]

eda = {
    "train_unique_query_texts": int(train.query_norm.nunique()),
    "train_unique_contexts": int(train.context_key.nunique()),
    "duplicate_context_item_pairs": int(train.duplicated(["context_key", "item_id"]).sum()),
    "benchmark_query_text_seen_fraction": float(queries.query_norm.isin(train.query_norm).mean()),
    "benchmark_item_seen_fraction": float(items.item_id.isin(train.item_id).mean()),
    "positive_same_location_fraction": float((train.search_location_id == train.item_location_id).mean()),
    "positive_same_category_fraction": float((train.search_category == train.item_category_id).mean()),
}
print(json.dumps(eda, ensure_ascii=False, indent=2))
print("\nДлины текстов корпуса, символы:")
print(items[["item_title_raw", "item_description_raw", "item_infm_params_text"]]
      .apply(lambda s: s.str.len().describe(percentiles=[.5, .95, .99])).round(1).to_string())


{
  "train_unique_query_texts": 73873,
  "train_unique_contexts": 354463,
  "duplicate_context_item_pairs": 30619,
  "benchmark_query_text_seen_fraction": 0.3735725938009788,
  "benchmark_item_seen_fraction": 0.09588186795763483,
  "positive_same_location_fraction": 0.8310195650557696,
  "positive_same_category_fraction": 0.9998935043693349
}

Длины текстов корпуса, символы:
       item_title_raw  item_description_raw  item_infm_params_text
count        189212.0              189212.0               189212.0
mean             33.6                1404.9                  973.9
std              12.2                1284.3                  875.9
min               1.0                   0.0                   10.0
50%              35.0                1054.0                  738.0
95%              50.0                4205.0                 2475.0
99%              53.0                5541.0                 4687.9
max             100.0                8494.0                19068.0


In [10]:
pairs = train.drop_duplicates(["context_key", "item_id"]).copy()
eligible = pairs[pairs.item_id.isin(ITEM_TO_ROW)].copy()
rng = np.random.default_rng(SEED)
text_pool = np.sort(eligible.query_norm.unique().astype(str))
selected_texts = rng.choice(text_pool, size=min(CONFIG["validation_texts"], len(text_pool)), replace=False)
selected = eligible[eligible.query_norm.isin(selected_texts)]
context_pool = selected[["query_norm", "context_key"]].drop_duplicates().sort_values(["query_norm", "context_key"])
chosen_keys = []
for _, group in context_pool.groupby("query_norm", sort=True):
    chosen_keys.append(group.context_key.iloc[int(rng.integers(len(group)))])
validation_pairs = selected[selected.context_key.isin(chosen_keys)]
validation = validation_pairs.drop_duplicates("context_key")[[*QUERY_COLS, "query_norm", "context_key"]].copy()
validation = validation.sort_values("context_key").reset_index(drop=True)
labels_by_key = validation_pairs.groupby("context_key").item_id.agg(lambda s: sorted(set(s)))
labels = [set(ITEM_TO_ROW[item_id] for item_id in labels_by_key[key]) for key in validation.context_key]
val_positive_ids = np.sort(validation_pairs.item_id.unique().astype(str))
cold_item_ids = set(rng.choice(val_positive_ids, size=int(CONFIG["cold_item_fraction"] * len(val_positive_ids)), replace=False))
fit_pairs = pairs[~pairs.query_norm.isin(selected_texts) & ~pairs.item_id.isin(cold_item_ids)].copy()
assert not set(validation.query_norm) & set(fit_pairs.query_norm)
assert not cold_item_ids & set(fit_pairs.item_id)
assert all(labels)
split_order = rng.permutation(len(validation))
dev_indices = np.sort(split_order[:int(len(validation) * CONFIG["development_fraction"])])
holdout_indices = np.sort(split_order[int(len(validation) * CONFIG["development_fraction"]):])
validation["split"] = "holdout"
validation.loc[dev_indices, "split"] = "development"
validation["relevant_item_ids"] = [" ".join(labels_by_key[key]) for key in validation.context_key]
validation.to_parquet(CACHE / "validation.parquet", index=False)
print("History pairs:", len(fit_pairs), "; validation queries:", len(validation))
print("Development:", len(dev_indices), "; holdout:", len(holdout_indices))
print("Mean positives:", round(np.mean([len(x) for x in labels]), 3))
print("Actual unseen positive item fraction:", round(1 - validation_pairs.item_id.isin(fit_pairs.item_id).mean(), 4))
# Полные тексты train больше не нужны; сохраняем компактную историю для обучения.
HISTORY_COLS = [*QUERY_COLS, "query_norm", "context_key", "item_id", "item_location_id", "item_microcat_id"]
history_all = pairs[HISTORY_COLS].copy()
history_fit = fit_pairs[HISTORY_COLS].copy()
del train, pairs, fit_pairs, eligible, selected, validation_pairs
gc.collect()


History pairs: 428634 ; validation queries: 1400
Development: 700 ; holdout: 700
Mean positives: 1.026
Actual unseen positive item fraction: 0.952


### Лексический индекс

Русская нормализация, стемминг и BM25 дают быстрый канал для точных
совпадений. Фильтры запроса индексируются отдельно от основного текста.

In [11]:
stemmer = snowballstemmer.stemmer("russian")
STOP_WORDS = set("и в во на с со к ко по из за от до для у о об а но или это как что я мы вы он она они не без при под над".split())

@lru_cache(maxsize=350_000)
def stem_word(word):
    return stemmer.stemWord(word)

def tokenize_words(text):
    return [stem_word(w) for w in re.findall(r"[a-zа-я0-9]+", normalize_text(text))
            if len(w) > 1 and w not in STOP_WORDS]

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

input_hashes = {name: sha256_file(ROOT / name) for name in
                ["train.parquet", "benchmark_items.parquet", "benchmark_queries.parquet"]}
# Frozen recipe identity for unchanged unsupervised indexes.
code_hash = '135258dc1337584d396735648624b90111b66492d7dc20b04ba55f99af154e7a'
fingerprint = hashlib.sha256(json.dumps({"inputs": input_hashes, "config": CONFIG,
                                        "ranker": RANKER_CONFIG, "semantic": SEMANTIC_CONFIG,
                                        "code": code_hash}, sort_keys=True).encode()).hexdigest()[:16]
index_path = CACHE / f"lexical_{fingerprint}.joblib"

def save_cache(value, path):
    # Atomic publication: an interrupted write cannot become a valid cache file.
    temporary = path.with_suffix(path.suffix + ".tmp")
    joblib.dump(value, temporary, compress=0)
    temporary.replace(path)

class BM25Index:
    def __init__(self, texts, max_features, k1=1.2, b=0.75):
        self.vectorizer = CountVectorizer(tokenizer=tokenize_words, token_pattern=None,
                                          lowercase=False, dtype=np.float32,
                                          max_features=max_features)
        counts = self.vectorizer.fit_transform(texts).tocsr()
        n = counts.shape[0]
        lengths = np.asarray(counts.sum(axis=1)).ravel()
        df = np.bincount(counts.indices, minlength=counts.shape[1]).astype(np.float32)
        self.idf = np.log1p((n - df + 0.5) / (df + 0.5)).astype(np.float32)
        norm = k1 * (1 - b + b * lengths / max(float(lengths.mean()), 1))
        row_norm = np.repeat(norm, np.diff(counts.indptr))
        counts.data = (counts.data * (k1 + 1) / (counts.data + row_norm) * self.idf[counts.indices]).astype(np.float32)
        self.matrix = counts
        self.transpose = counts.T.tocsr()

    def query_matrix(self, texts):
        matrix = self.vectorizer.transform(texts).tocsr()
        matrix.data[:] = 1  # repeated query words do not increase importance
        return matrix

started = time.perf_counter()
if USE_CACHE and index_path.exists():
    lexical = joblib.load(index_path)
    print("Loaded lexical cache")
else:
    titles = items.item_title_raw.map(normalize_text).tolist()
    bodies = (items.item_title_raw + " " + items.item_infm_params_text.str.slice(0, CONFIG["params_chars"])
              + " " + items.item_description_raw.str.slice(0, CONFIG["description_chars"])).tolist()
    print("Building title BM25...", flush=True)
    title_index = BM25Index(titles, CONFIG["word_features"])
    print("Building body BM25...", flush=True)
    body_index = BM25Index(bodies, CONFIG["word_features"])
    del bodies
    print("Building character title index...", flush=True)
    char_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                                     max_features=CONFIG["char_features"], sublinear_tf=True,
                                     dtype=np.float32)
    char_matrix = char_vectorizer.fit_transform(titles).tocsr()
    lexical = {"title": title_index, "body": body_index, "char_vectorizer": char_vectorizer,
               "char_transpose": char_matrix.T.tocsr()}
    save_cache(lexical, index_path)
    del titles, char_matrix, title_index, body_index, char_vectorizer
    gc.collect()
print("Index preparation seconds:", round(time.perf_counter() - started, 1))


Loaded lexical cache
Index preparation seconds: 1.1


### Корпусные массивы и географическая совместимость

Числовые поля и координаты преобразуются в плотные массивы, чтобы тысячи
объявлений одного запроса оценивались пакетно.

In [12]:
ITEM_LOCS = items.item_location_id.to_numpy()
ITEM_MICROS = items.item_microcat_id.to_numpy()
MICROS = np.sort(items.item_microcat_id.unique())
MICRO_TO_COL = {int(value): i for i, value in enumerate(MICROS)}
ITEM_MICRO_COLS = np.array([MICRO_TO_COL[int(x)] for x in ITEM_MICROS])
centers = items.groupby("item_location_id")[["item_latitude", "item_longitude"]].median()
item_lat = np.radians(items.item_latitude.to_numpy())
item_lon = np.radians(items.item_longitude.to_numpy())

def stable_topk(scores, k=50):
    """Descending score, then ascending item row (= ascending item_id)."""
    scores = np.nan_to_num(np.asarray(scores), nan=-np.inf)
    k = min(k, len(scores))
    threshold = np.partition(scores, len(scores) - k)[len(scores) - k]
    above = np.flatnonzero(scores > threshold)
    tied = np.flatnonzero(scores == threshold)[:k - len(above)]
    chosen = np.concatenate([above, tied])
    return chosen[np.lexsort((chosen, -scores[chosen]))]

class HistorySignals:
    def __init__(self, history):
        transitions = history.groupby(["search_location_id", "item_location_id"]).size()
        self.transitions = {}
        for search_loc, counts in transitions.groupby(level=0):
            # Shrink small-sample transitions; do not hard-filter to observed cities.
            maximum = max(float(counts.max()), 1)
            strength = min(1.0, float(counts.sum()) / 20)
            self.transitions[int(search_loc)] = {
                int(pair[1]): float(np.sqrt(count / maximum) * strength)
                for pair, count in counts.items()
            }
        texts = np.sort(history.query_norm.unique().astype(str))
        self.texts = texts
        self.text_to_row = {text: i for i, text in enumerate(texts)}
        self.vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                                         min_df=2, max_features=180_000, sublinear_tf=True,
                                         dtype=np.float32)
        self.text_matrix_t = self.vectorizer.fit_transform(texts).T.tocsr()
        valid = history[history.item_microcat_id.isin(MICRO_TO_COL)]
        counts = valid.groupby(["query_norm", "item_microcat_id"]).size()
        rows = [self.text_to_row[text] for text, _ in counts.index]
        columns = [MICRO_TO_COL[int(micro)] for _, micro in counts.index]
        self.micro_matrix = sp.csr_matrix((counts.to_numpy(dtype=np.float32), (rows, columns)),
                                          shape=(len(texts), len(MICROS)))
        sums = np.asarray(self.micro_matrix.sum(axis=1)).ravel()
        self.micro_matrix = sp.diags(1 / np.maximum(sums, 1)) @ self.micro_matrix

    def micro_predictions(self, query_texts):
        query_matrix = self.vectorizer.transform(query_texts)
        outputs = []
        for row in range(query_matrix.shape[0]):
            similarities = (query_matrix[row] @ self.text_matrix_t).toarray().ravel()
            nearest = stable_topk(similarities, 20)
            weights = np.maximum(similarities[nearest] - 0.2, 0) ** 3
            # Exact known query carries more evidence than several weak neighbours.
            exact = self.text_to_row.get(query_texts[row])
            if exact is not None:
                nearest = np.array([exact])
                weights = np.array([1.0], dtype=np.float32)
            probabilities = np.asarray(self.micro_matrix[nearest].T @ weights).ravel()
            if probabilities.max() > 0:
                probabilities /= probabilities.max()
            else:
                probabilities[:] = 1
            outputs.append(probabilities.astype(np.float32))
        return np.stack(outputs)

    def geography(self, search_location):
        result = np.zeros(len(items), dtype=np.float32)
        if search_location in centers.index:
            lat, lon = np.radians(centers.loc[search_location].to_numpy(dtype=float))
            hav = np.sin((item_lat - lat) / 2)**2 + np.cos(lat) * np.cos(item_lat) * np.sin((item_lon - lon) / 2)**2
            distance = 6371 * 2 * np.arcsin(np.sqrt(np.clip(hav, 0, 1)))
            result = np.nan_to_num(0.7 * np.exp(-distance / 40), nan=0).astype(np.float32)
        mapping = self.transitions.get(int(search_location), {})
        if mapping:
            unique_locs, inverse = np.unique(ITEM_LOCS, return_inverse=True)
            transition = np.array([mapping.get(int(loc), 0) for loc in unique_locs], dtype=np.float32)[inverse]
            result = np.maximum(result, transition)
        result[ITEM_LOCS == search_location] = 1
        if result.max() == 0:
            result[:] = 1
        return result

def select_query_contexts(history, count, seed):
    eligible_pairs = history[history.item_id.isin(ITEM_TO_ROW)]
    local_rng = np.random.default_rng(seed)
    texts = np.sort(eligible_pairs.query_norm.unique().astype(str))
    chosen_texts = local_rng.choice(texts, min(count, len(texts)), replace=False)
    contexts = eligible_pairs[eligible_pairs.query_norm.isin(chosen_texts)].drop_duplicates("context_key")
    chosen = []
    for _, group in contexts.sort_values("context_key").groupby("query_norm", sort=True):
        chosen.append(group.iloc[int(local_rng.integers(len(group)))])
    return pd.DataFrame(chosen)[[*QUERY_COLS, "query_norm", "context_key"]].sort_values("context_key").reset_index(drop=True)

def query_labels(history, query_frame):
    pairs = history[history.context_key.isin(query_frame.context_key) & history.item_id.isin(ITEM_TO_ROW)]
    grouped = pairs.groupby("context_key").item_id.agg(lambda s: set(ITEM_TO_ROW[x] for x in s))
    return [grouped.get(key, set()) for key in query_frame.context_key]

def purge_history(history, query_frame, relevant, seed):
    positives = np.array(sorted({ITEM_IDS[i] for truth in relevant for i in truth}))
    local_rng = np.random.default_rng(seed)
    cold_ids = set(local_rng.choice(positives, int(CONFIG["cold_item_fraction"] * len(positives)), replace=False))
    clean = history[~history.query_norm.isin(query_frame.query_norm) & ~history.item_id.isin(cold_ids)].copy()
    assert not set(query_frame.query_norm) & set(clean.query_norm)
    assert not cold_ids & set(clean.item_id)
    return clean, cold_ids

audit_queries = select_query_contexts(history_fit, RANKER_CONFIG["audit_queries"], SEED + 101)
audit_labels = query_labels(history_fit, audit_queries)
ranker_history, audit_cold_ids = purge_history(history_fit, audit_queries, audit_labels, SEED + 102)
original_training_queries = select_query_contexts(ranker_history, RANKER_CONFIG["training_queries"], SEED + 103)
# These 600 texts were not direct training queries in the already inspected v0.2
# model. Remove their labels from all history before fitting the new comparison.
fresh_source = ranker_history[~ranker_history.query_norm.isin(original_training_queries.query_norm)]
fresh_queries = select_query_contexts(fresh_source, RANKER_CONFIG["fresh_audit_queries"], SEED + 501)
fresh_labels = query_labels(ranker_history, fresh_queries)
ranker_history, fresh_cold_ids = purge_history(ranker_history, fresh_queries, fresh_labels, SEED + 502)
training_queries = select_query_contexts(ranker_history, RANKER_CONFIG["training_queries"], SEED + 103)
assert all(fresh_labels)
assert not set(fresh_queries.query_norm) & set(original_training_queries.query_norm)
assert not set(fresh_queries.query_norm) & set(training_queries.query_norm)
assert not set(fresh_queries.query_norm) & set(validation.query_norm)
assert not set(fresh_queries.query_norm) & set(audit_queries.query_norm)
assert not set(fresh_queries.query_norm) & set(ranker_history.query_norm)
assert not fresh_cold_ids & set(ranker_history.item_id)
fresh_protocol = {"fresh_queries": len(fresh_queries), "development_queries": len(validation) + len(audit_queries),
    "training_queries": len(training_queries), "query_text_overlap": 0, "purged_item_overlap": 0,
    "old_training_query_text_overlap": 0, "purged_positive_items": len(fresh_cold_ids),
    "used_for_model_selection": False}
(CACHE / "fresh_audit_protocol.json").write_text(json.dumps(fresh_protocol, indent=2), encoding="utf-8")
del fresh_source

started = time.perf_counter()
history_model = HistorySignals(ranker_history)
print("History signals prepared in", round(time.perf_counter() - started, 1), "seconds")


History signals prepared in 1.5 seconds


In [13]:
def normalized_scores(scores):
    maximum = float(scores.max())
    return scores / maximum if maximum > 0 else scores

def retrieve_features(query_frame, history_model, progress_every=200):
    texts = query_frame.query_norm.astype(str).tolist()
    title_queries = lexical["title"].query_matrix(texts)
    body_queries = lexical["body"].query_matrix(texts)
    filter_queries = lexical["body"].query_matrix(query_frame.search_infm_params_text.astype(str).tolist())
    char_queries = lexical["char_vectorizer"].transform(texts)
    micro_predictions = history_model.micro_predictions(texts)
    outputs = []
    geo_cache = {}
    started = time.perf_counter()
    for i, query in enumerate(query_frame.itertuples(index=False)):
        title = normalized_scores((title_queries[i] @ lexical["title"].transpose).toarray().ravel())
        body = normalized_scores((body_queries[i] @ lexical["body"].transpose).toarray().ravel())
        chars = normalized_scores((char_queries[i] @ lexical["char_transpose"]).toarray().ravel())
        filters = normalized_scores((filter_queries[i] @ lexical["body"].transpose).toarray().ravel())
        loc = int(query.search_location_id)
        if loc not in geo_cache:
            geo_cache[loc] = history_model.geography(loc)
        geo = geo_cache[loc]
        micro = micro_predictions[i][ITEM_MICRO_COLS]
        sources = [title, body, chars]
        candidate_parts = []
        for source in sources:
            candidate_parts.append(stable_topk(source, CONFIG["retrieval_pool"]))
            candidate_parts.append(stable_topk(source * (0.03 + 0.97 * geo), CONFIG["retrieval_pool"]))
        mixed = (0.5 * title + 0.3 * body + 0.2 * chars)
        candidate_parts.append(stable_topk(mixed * (0.03 + 0.97 * geo) * (0.2 + 0.8 * micro), CONFIG["retrieval_pool"]))
        candidates = np.unique(np.concatenate(candidate_parts))
        features = np.column_stack([title[candidates], body[candidates], chars[candidates],
                                    filters[candidates], geo[candidates], micro[candidates]]).astype(np.float32)
        outputs.append((candidates, features))
        if (i + 1) % progress_every == 0 or i + 1 == len(query_frame):
            print(f"Retrieved {i + 1}/{len(query_frame)} queries; {time.perf_counter() - started:.1f}s", flush=True)
    return outputs

def score_candidates(features, config):
    title, body, chars, filters, geo, micro = features.T
    score = config["title"] * title + config["body"] * body + config["char"] * chars + config["filter"] * filters
    score *= (0.03 + 0.97 * geo) ** config["geo"]
    score *= (0.2 + 0.8 * micro) ** config["micro"]
    return score

def predict_from_features(records, config, k=50):
    return [candidates[stable_topk(score_candidates(features, config), k)] for candidates, features in records]

def per_query_recall(predictions, relevant):
    assert len(predictions) == len(relevant)
    return np.array([len(set(map(int, predicted)) & truth) / len(truth)
                     for predicted, truth in zip(predictions, relevant)], dtype=float)
best_config = {'title':.3,'body':.5,'char':.2,'filter':.03,'geo':.5,'micro':.5}


### Признаки и набор трудных конкурентов

Текстовое покрытие, цена, рейтинг, расстояние, ранги поиска и источник
попадания в пул передаются ранкеру. Случайные кандидаты сохраняют широту
обучения, трудные приближают его к границе top-50.

In [14]:
RANK_FEATURE_NAMES = [
    "title_bm25", "body_bm25", "char_tfidf", "filter_bm25", "geo_compatibility", "micro_compatibility",
    "baseline_score", "log_baseline_rank", "log_title_rank", "log_body_rank", "log_char_rank",
    "same_location", "log_distance_km", "title_token_coverage", "body_token_coverage",
    "title_idf_coverage", "body_idf_coverage", "query_in_title", "query_equals_title",
    "query_characters", "query_tokens", "log_filter_characters", "rating_required", "rating_gap",
    "rating", "log_reviews", "log_price", "phone_hidden", "messages_forbidden",
    "log_title_characters", "log_description_characters", "log_params_characters",
]
assert not any("category" in name or name.endswith("_id") for name in RANK_FEATURE_NAMES)
normalized_titles = items.item_title_raw.map(normalize_text).to_numpy(dtype=str)
item_quality = np.column_stack([
    items.item_rating.to_numpy(dtype=np.float32),
    np.log1p(items.item_rating_reviews_count.clip(lower=0).to_numpy(dtype=np.float32)),
    np.log1p(items.item_price.clip(lower=0).to_numpy(dtype=np.float32)),
    items.item_is_phone_hidden.to_numpy(dtype=np.float32),
    items.item_is_message_forbidden.to_numpy(dtype=np.float32),
    np.log1p(items.item_title_raw.str.len().to_numpy(dtype=np.float32)),
    np.log1p(items.item_description_raw.str.len().to_numpy(dtype=np.float32)),
    np.log1p(items.item_infm_params_text.str.len().to_numpy(dtype=np.float32)),
]).astype(np.float32)

def token_coverage(index, candidates, terms):
    vocabulary = index.vectorizer.vocabulary_
    indices = [vocabulary[term] for term in terms if term in vocabulary]
    if not indices:
        return np.zeros((len(candidates), 2), dtype=np.float32)
    present = (index.matrix[candidates][:, indices].toarray() > 0).astype(np.float32)
    idf = index.idf[indices]
    unknown = len(terms) - len(indices)
    denominator = float(idf.sum()) + unknown * float(index.idf.max())
    return np.column_stack([present.sum(axis=1) / max(len(terms), 1),
                            present @ idf / max(denominator, 1e-6)])

def rank_features(query, record):
    candidates, base = record
    n = len(candidates)
    baseline = score_candidates(base, best_config)
    ranks = np.column_stack([np.log1p(rankdata(-values, method="min")) for values in
                             [baseline, base[:, 0], base[:, 1], base[:, 2]]])
    same_location = (ITEM_LOCS[candidates] == int(query.search_location_id)).astype(np.float32)
    distance = np.full(n, np.nan, dtype=np.float32)
    if query.search_location_id in centers.index:
        lat, lon = np.radians(centers.loc[query.search_location_id].to_numpy(dtype=float))
        hav = np.sin((item_lat[candidates] - lat) / 2)**2 + np.cos(lat) * np.cos(item_lat[candidates]) * np.sin((item_lon[candidates] - lon) / 2)**2
        distance = np.log1p(6371 * 2 * np.arcsin(np.sqrt(np.clip(hav, 0, 1))))
    terms = sorted(set(tokenize_words(query.query_norm)))
    title_coverage = token_coverage(lexical["title"], candidates, terms)
    body_coverage = token_coverage(lexical["body"], candidates, terms)
    title_texts = normalized_titles[candidates]
    substring = np.array([query.query_norm in text for text in title_texts], dtype=np.float32)
    exact = (title_texts == query.query_norm).astype(np.float32)
    threshold_match = re.search(r"рейтинг[^\d]{0,40}([0-5])", normalize_text(query.search_infm_params_text))
    required_rating = float(threshold_match.group(1)) if threshold_match else np.nan
    query_constants = np.tile([len(query.query_norm), len(terms), np.log1p(len(query.search_infm_params_text)), required_rating], (n, 1))
    result = np.column_stack([
        base, baseline, ranks, same_location, distance,
        title_coverage[:, 0], body_coverage[:, 0], title_coverage[:, 1], body_coverage[:, 1],
        substring, exact, query_constants, item_quality[candidates, 0] - required_rating,
        item_quality[candidates],
    ]).astype(np.float32)
    assert result.shape == (n, 32)
    assert not np.isinf(result).any()
    return result

def hard_negative_sample(candidates, base, truth, local_rng):
    """Keep retrieved positives, hard negatives and a random tail; never inject labels."""
    target = np.isin(candidates, list(truth))
    positive = np.flatnonzero(target)
    negative = np.flatnonzero(~target)
    if not len(positive) or not len(negative):
        return np.array([], dtype=int), target
    ordered = negative[stable_topk(score_candidates(base[negative], best_config), len(negative))]
    hard = ordered[:RANKER_CONFIG["hard_negatives"]]
    tail = ordered[len(hard):]
    random_tail = local_rng.choice(tail, min(len(tail), RANKER_CONFIG["random_negatives"]), replace=False)
    selected = np.sort(np.concatenate([positive, hard, random_tail]))
    assert not set(candidates[selected[~target[selected]]]) & truth
    return selected, target

def build_oof_training(history, query_frame, stage):
    path = CACHE / f"{stage}_training_{fingerprint}.joblib"
    if USE_CACHE and path.exists():
        return joblib.load(path)
    truths = query_labels(history, query_frame)
    assert all(truths)
    local_rng = np.random.default_rng(SEED + 200)
    assignments = np.empty(len(query_frame), dtype=int)
    assignments[local_rng.permutation(len(query_frame))] = np.arange(len(query_frame)) % RANKER_CONFIG["folds"]
    xs, ys, weights, groups, audit = [], [], [], [], []
    for fold in range(RANKER_CONFIG["folds"]):
        positions = np.flatnonzero(assignments == fold)
        fold_queries = query_frame.iloc[positions]
        fold_truths = [truths[i] for i in positions]
        clean_history, cold_ids = purge_history(history, fold_queries, fold_truths, SEED + 300 + fold)
        print(f"{stage}: fold {fold + 1}/{RANKER_CONFIG['folds']}; queries={len(positions)}; history={len(clean_history)}", flush=True)
        fold_history_model = HistorySignals(clean_history)
        records = retrieve_features(fold_queries, fold_history_model, progress_every=500)
        missing = 0
        for j, (query, record, truth) in enumerate(zip(fold_queries.itertuples(index=False), records, fold_truths)):
            candidates, base = record
            selected, targets = hard_negative_sample(candidates, base, truth, local_rng)
            if not len(selected):
                missing += 1
                continue
            # Ranks are computed against the full pool before negative sampling.
            features = rank_features(query, record)[selected]
            y = targets[selected].astype(np.uint8)
            n_positive, n_negative = int(y.sum()), int((1 - y).sum())
            # Every query has the same total loss weight; its positives share weight.
            w = np.where(y == 1, 100.0 / n_positive, 100.0 / n_negative).astype(np.float32)
            xs.append(features)
            ys.append(y)
            weights.append(w)
            groups.append(np.full(len(selected), positions[j], dtype=np.int32))
        audit.append({"fold": fold, "queries": len(positions), "history_pairs": len(clean_history),
                      "purged_positive_item_ids": len(cold_ids), "query_text_overlap": 0,
                      "purged_item_overlap": 0, "queries_without_retrieved_positive": missing,
                      "pool_recall": float(per_query_recall([r[0] for r in records], fold_truths).mean())})
        del fold_history_model, clean_history, records
        gc.collect()
    dataset = {"X": np.concatenate(xs), "y": np.concatenate(ys), "weight": np.concatenate(weights),
               "group": np.concatenate(groups), "audit": audit}
    dataset["weight"] /= dataset["weight"].mean()
    assert len(np.unique(dataset["group"])) == sum(x["queries"] - x["queries_without_retrieved_positive"] for x in audit)
    save_cache(dataset, path)
    (CACHE / f"{stage}_leakage_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
    print(stage, "training matrix:", dataset["X"].shape, "; positives:", int(dataset["y"].sum()))
    return dataset


def build_rank_feature_records(query_frame, records, stage):
    path = CACHE / f"{stage}_rank_features_{fingerprint}.joblib"
    if USE_CACHE and path.exists():
        return joblib.load(path)
    output = [rank_features(query, record) for query, record in zip(query_frame.itertuples(index=False), records)]
    save_cache(output, path)
    return output

def model_scores(model, feature_records):
    sizes = [len(x) for x in feature_records]
    # Larger batches avoid repeated OpenMP startup for every individual query.
    output = []
    for start in range(0, len(feature_records), 64):
        block = feature_records[start:start + 64]
        scores = model.decision_function(np.concatenate(block))
        output.extend(np.split(scores, np.cumsum(sizes[start:start + len(block)])[:-1]))
    return output

def blend_predictions(records, learned_scores, alpha, k=50):
    if alpha == 0:
        return predict_from_features(records, best_config, k)
    predictions = []
    for (candidates, features), learned in zip(records, learned_scores):
        if alpha == 1:
            score = learned
        else:
            baseline = score_candidates(features, best_config)
            score = alpha / (60 + rankdata(-learned, method="min")) + (1 - alpha) / (60 + rankdata(-baseline, method="min"))
        predictions.append(candidates[stable_topk(score, k)])
    return predictions

def selected_predictions(query_frame, records, model, stage):
    if ranker_choice["alpha"] == 0:
        return predict_from_features(records, best_config)
    enriched = build_rank_feature_records(query_frame, records, stage)
    scores = model_scores(model, enriched)
    return blend_predictions(records, scores, ranker_choice["alpha"])


### Семантический поиск E5

Локальный encoder сопоставляет короткий запрос с названием, параметрами и
описанием объявления. Векторы документов сохранены; код не обращается к
сетевым сервисам при воспроизведении.

In [15]:
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
if (ROOT / ".semantic_deps").is_dir():
    sys.path.insert(0, str(ROOT / ".semantic_deps"))
import torch
import transformers
from transformers import AutoModel, AutoTokenizer

torch.set_num_threads(8)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DIR = ROOT / "models" / "multilingual-e5-small"
if not (MODEL_DIR / "manifest.json").exists():
    raise FileNotFoundError("Prepare models/multilingual-e5-small before offline execution; see README")
model_manifest = json.loads((MODEL_DIR / "manifest.json").read_text(encoding="utf-8"))
assert model_manifest["revision"] == SEMANTIC_CONFIG["revision"]
for name, expected in model_manifest["sha256"].items():
    assert (MODEL_DIR / name).exists(), f"Missing local model file: {name}"
    assert sha256_file(MODEL_DIR / name) == expected, f"Modified model file: {name}"

def save_array(array, path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as stream:
        np.save(stream, array, allow_pickle=False)
    temporary.replace(path)

class SemanticIndex:
    def __init__(self):
        self.tokenizer = None
        self.encoder = None

    def encode(self, texts, prefix):
        if self.encoder is None:
            self.tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True, use_fast=True)
            self.encoder = AutoModel.from_pretrained(MODEL_DIR, local_files_only=True,
                                                     attn_implementation="eager").to(DEVICE).eval()
        # A stable length order reduces padding without changing document IDs.
        order = np.argsort(np.array([len(text) for text in texts]), kind="stable")
        output = np.empty((len(texts), 384), dtype=np.float32)
        started = time.perf_counter()
        with torch.inference_mode():
            for start in range(0, len(order), SEMANTIC_CONFIG["batch_size"]):
                positions = order[start:start + SEMANTIC_CONFIG["batch_size"]]
                batch = self.tokenizer([prefix + texts[i] for i in positions], padding=True,
                    truncation=True, max_length=SEMANTIC_CONFIG["max_length"], return_tensors="pt")
                batch = {key: value.to(DEVICE) for key, value in batch.items()}
                hidden = self.encoder(**batch).last_hidden_state
                mask = batch["attention_mask"].unsqueeze(-1)
                pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
                embeddings = torch.nn.functional.normalize(pooled, p=2, dim=1)
                output[positions] = embeddings.cpu().numpy()
                completed = min(start + len(positions), len(order))
                if completed % (SEMANTIC_CONFIG["batch_size"] * 50) == 0 or completed == len(order):
                    elapsed = time.perf_counter() - started
                    print(f"E5 {prefix.strip()} {completed}/{len(order)}; {elapsed:.1f}s; {completed / max(elapsed, .01):.0f} texts/s", flush=True)
        assert np.isfinite(output).all()
        assert np.allclose(np.linalg.norm(output, axis=1), 1, atol=2e-5)
        return output

    def prepare(self, query_texts):
        documents = (items.item_title_raw + ". " + items.item_infm_params_text.str.slice(0, SEMANTIC_CONFIG["params_chars"])
                     + ". " + items.item_description_raw.str.slice(0, SEMANTIC_CONFIG["description_chars"]))
        # Encoder caches depend on text, model, input order and encoding parameters,
        # rather than the ranker code. AVITO_REBUILD_CACHE=1 still rebuilds everything.
        payload = {"model": model_manifest, "encoding": SEMANTIC_CONFIG, "items": input_hashes["benchmark_items.parquet"]}
        key = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:16]
        self.item_path = CACHE / f"e5_items_{key}.npy"
        if USE_CACHE and self.item_path.exists():
            self.items = np.load(self.item_path, allow_pickle=False)
            print("Loaded E5 document embeddings")
        else:
            self.items = self.encode(documents.astype(str).tolist(), "passage: ")
            save_array(self.items, self.item_path)
        texts = sorted(set(map(str, query_texts)))
        text_hash = hashlib.sha256(json.dumps(texts, ensure_ascii=False).encode()).hexdigest()[:16]
        self.query_path = CACHE / f"e5_queries_{key}_{text_hash}.npy"
        if USE_CACHE and self.query_path.exists():
            self.queries = np.load(self.query_path, allow_pickle=False)
        else:
            self.queries = self.encode(texts, "query: ")
            save_array(self.queries, self.query_path)
        self.query_to_row = {text: i for i, text in enumerate(texts)}
        assert self.items.shape == (len(items), 384)
        assert self.queries.shape == (len(texts), 384)
        del self.encoder, self.tokenizer
        self.encoder = self.tokenizer = None
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        self.item_transpose = self.items.astype(np.float64).T

    def scores(self, texts):
        # Also support new queries outside the notebook's prepared evaluation set.
        missing = sorted(set(map(str, texts)) - self.query_to_row.keys())
        if missing:
            additional = self.encode(missing, "query: ")
            offset = len(self.queries)
            self.queries = np.concatenate([self.queries, additional])
            self.query_to_row.update({text: offset + i for i, text in enumerate(missing)})
        vectors = self.queries[[self.query_to_row[str(text)] for text in texts]]
        return (vectors.astype(np.float64) @ self.item_transpose).astype(np.float32)

semantic_index = SemanticIndex()


### Объединение каналов кандидатного пула

Лексические и семантические находки дополняют друг друга. Полнота широкого
пула на локальной валидации около 0.995; объявление вне пула ранкер вернуть
не сможет.

In [16]:
legacy_retrieve_features = retrieve_features
legacy_rank_features = rank_features
legacy_score_candidates = score_candidates
legacy_negative_sample = hard_negative_sample
LEGACY_FEATURE_NAMES = list(RANK_FEATURE_NAMES)
SEMANTIC_FEATURE_NAMES = ['e5_cosine','e5_geo_affinity','from_lexical_pool','from_semantic_pool','log_e5_rank','log_e5_geo_rank']
RANK_FEATURE_NAMES += SEMANTIC_FEATURE_NAMES
def semantic_affinity(cosine, geography):
    return np.exp((cosine - 1) / SEMANTIC_CONFIG["temperature"]) * (0.03 + 0.97 * geography)

def retrieve_semantic_features(query_frame, history_model, progress_every=200):
    texts = query_frame.query_norm.astype(str).tolist()
    title_queries = lexical["title"].query_matrix(texts)
    body_queries = lexical["body"].query_matrix(texts)
    filter_queries = lexical["body"].query_matrix(query_frame.search_infm_params_text.astype(str).tolist())
    char_queries = lexical["char_vectorizer"].transform(texts)
    micro_predictions = history_model.micro_predictions(texts)
    outputs, geo_cache = [], {}
    started = time.perf_counter()
    dense_block = None
    for i, query in enumerate(query_frame.itertuples(index=False)):
        if i % 64 == 0:
            dense_block = semantic_index.scores(texts[i:i + 64])
        cosine = dense_block[i % 64]
        title = normalized_scores((title_queries[i] @ lexical["title"].transpose).toarray().ravel())
        body = normalized_scores((body_queries[i] @ lexical["body"].transpose).toarray().ravel())
        chars = normalized_scores((char_queries[i] @ lexical["char_transpose"]).toarray().ravel())
        filters = normalized_scores((filter_queries[i] @ lexical["body"].transpose).toarray().ravel())
        loc = int(query.search_location_id)
        if loc not in geo_cache:
            geo_cache[loc] = history_model.geography(loc)
        geo = geo_cache[loc]
        micro = micro_predictions[i][ITEM_MICRO_COLS]
        candidate_parts = []
        for source in [title, body, chars]:
            candidate_parts.append(stable_topk(source, CONFIG["retrieval_pool"]))
            candidate_parts.append(stable_topk(source * (0.03 + 0.97 * geo), CONFIG["retrieval_pool"]))
        mixed = 0.5 * title + 0.3 * body + 0.2 * chars
        candidate_parts.append(stable_topk(mixed * (0.03 + 0.97 * geo) * (0.2 + 0.8 * micro), CONFIG["retrieval_pool"]))
        lexical_candidates = np.unique(np.concatenate(candidate_parts))
        affinity = semantic_affinity(cosine, geo)
        semantic_candidates = np.union1d(stable_topk(cosine, SEMANTIC_CONFIG["retrieval_pool"]),
                                          stable_topk(affinity, SEMANTIC_CONFIG["retrieval_pool"]))
        candidates = np.union1d(lexical_candidates, semantic_candidates)
        features = np.column_stack([title[candidates], body[candidates], chars[candidates],
            filters[candidates], geo[candidates], micro[candidates], cosine[candidates], affinity[candidates],
            np.isin(candidates, lexical_candidates), np.isin(candidates, semantic_candidates)]).astype(np.float32)
        outputs.append((candidates, features))
        if (i + 1) % progress_every == 0 or i + 1 == len(query_frame):
            print(f"Hybrid retrieved {i + 1}/{len(query_frame)}; {time.perf_counter() - started:.1f}s", flush=True)
    return outputs

def score_lexical_columns(features, config):
    return legacy_score_candidates(features[:, :6], config)

def rank_semantic_features(query, record):
    candidates, features = record
    original = legacy_rank_features(query, (candidates, features[:, :6]))
    semantic = features[:, 6:10]
    ranks = np.column_stack([np.log1p(rankdata(-values, method="min"))
                             for values in [semantic[:, 0], semantic[:, 1]]])
    result = np.column_stack([original, semantic, ranks]).astype(np.float32)
    assert result.shape == (len(candidates), len(RANK_FEATURE_NAMES))
    return result

def semantic_negative_sample(candidates, features, truth, local_rng):
    target = np.isin(candidates, list(truth))
    positive, negative = np.flatnonzero(target), np.flatnonzero(~target)
    if not len(positive) or not len(negative):
        return np.array([], dtype=int), target
    half = RANKER_CONFIG["hard_negatives"] // 2
    lexical_hard = negative[stable_topk(score_candidates(features[negative], best_config), half)]
    semantic_hard = negative[stable_topk(features[negative, 7], half)]
    hard = np.union1d(lexical_hard, semantic_hard)
    tail = np.setdiff1d(negative, hard)
    random_tail = local_rng.choice(tail, min(len(tail), RANKER_CONFIG["random_negatives"]), replace=False)
    selected = np.sort(np.concatenate([positive, hard, random_tail]))
    assert not set(candidates[selected[~target[selected]]]) & truth
    return selected, target

def cached_hybrid_records(query_frame, history, stage):
    path = CACHE / f"{stage}_hybrid_{fingerprint}.joblib"
    if USE_CACHE and path.exists():
        return joblib.load(path)
    # Always measure the expanded pool, even if selection falls back to lexical.
    records = retrieve_semantic_features(query_frame, history)
    save_cache(records, path)
    return records
retrieve_features = retrieve_semantic_features
rank_features = rank_semantic_features
score_candidates = score_lexical_columns
hard_negative_sample = semantic_negative_sample


## 3. История, география и подкатегория

Для каждого кандидата считаются частота взаимодействий, уверенность истории,
связь запроса с подкатегорией и локацией, расстояние до исполнителя и
доступность выездной или дистанционной услуги. Пропуски остаются явными,
чтобы ранкер сам выбирал, насколько доверять каждому сигналу.

In [17]:
from types import MappingProxyType
# Use the project's local dependency folder when present; otherwise requirements.txt.
if (ROOT / '.ranking_deps').is_dir():
    sys.path.insert(0, str(ROOT / '.ranking_deps'))
import lightgbm as lgb
INDEX_CACHE = CACHE
OLD_CACHE = ROOT / 'artifacts/ranking-v1'
CACHE = ROOT / 'artifacts/ranking-v5'
CACHE.mkdir(exist_ok=True)
V5_CONFIG = {'data_revision': 1, 'chunk': 128, 'folds': 3, 'cold_fraction': .9,
    'hard_lexical': 100, 'hard_geo_semantic': 100, 'hard_semantic': 60,
    'random_tail': 64, 'mined_hard': 160, 'trees': 400}
fingerprint = hashlib.sha256(json.dumps({'inputs': input_hashes, 'config': V5_CONFIG,
    'protocol': '0fb478413a4928393ddd87349be798ec6b703f6e0021f0339443a4ed605d0434',
    'pipeline': 'f92b61f6e92cd3893c8043e95edabd7f9bb6985998e31024107226be193b400b'}, sort_keys=True).encode()).hexdigest()[:16]
gold = freeze_gold(history_all, ITEM_TO_ROW)
PROTOCOL_CACHE = ROOT / 'artifacts/validation-v5'
training = pd.read_parquet(PROTOCOL_CACHE/'training_contexts.parquet').drop(columns='relevant_item_ids')
development = pd.read_parquet(PROTOCOL_CACHE/'development_contexts.parquet').drop(columns='relevant_item_ids')
control = pd.read_parquet(PROTOCOL_CACHE/'held_control_contexts.parquet').drop(columns='relevant_item_ids')
all_evaluation = pd.concat([development, control], ignore_index=True)
all_evaluation_truths = labels_from_gold(gold, all_evaluation)
training_history, evaluation_cold, _ = history_for_queries(history_all, all_evaluation,
    all_evaluation_truths, ITEM_IDS, 'unseen_text', .9, SEED+802)
assert not set(training.query_norm) & set(all_evaluation.query_norm)
assert all(labels_from_gold(gold, training))
known_target = float(queries.query_norm.isin(history_all.query_norm).mean())

# Reuse content-addressed document vectors; encode only this run's query set.
for file in INDEX_CACHE.glob('e5_items_*.npy'):
    target = CACHE/file.name
    if not target.exists():
        os.link(file, target)
semantic_index.prepare(pd.concat([training.query_norm, development.query_norm,
    control.query_norm, queries.query_norm, history_all.loc[history_all.item_id.isin(ITEM_TO_ROW),'query_norm']]))
EXTRA_FEATURE_NAMES = ['history_query_known', 'history_query_log_pairs',
    'history_query_micro_entropy', 'history_query_top_micro_probability',
    'geo_center_available', 'geo_transition_probability', 'geo_transition_log_support']
V5_FEATURE_NAMES = list(RANK_FEATURE_NAMES) + EXTRA_FEATURE_NAMES

# Verify the declared frozen components before using their scores or mining.
weights_path = CACHE / 'weights_manifest.json'
if weights_path.exists():
    supplied_weights = json.loads(weights_path.read_text(encoding='utf-8'))
    assert supplied_weights['input_sha256'] == input_hashes
    for relative, expected in supplied_weights['frozen_components'].items():
        assert sha256_file(ROOT / relative) == expected, f'Modified frozen component: {relative}'
    if USE_CACHE:
        for name, expected in supplied_weights['final_model'].items():
            assert sha256_file(CACHE / name) == expected, f'Modified final model: {name}'
        for name, expected in supplied_weights['embedding_sha256'].items():
            assert sha256_file(CACHE / name) == expected, f'Modified embedding cache: {name}'


Loaded E5 document embeddings


### Вектор признаков пары «запрос — объявление»

История добавляется к текстовым и географическим сигналам. Объявление,
которое не выбирали в логе, не считается доказанно нерелевантным.

In [18]:
class V5History(HistorySignals):
    """Base retrieval statistics plus explicit evidence strength for ranking."""
    def __init__(self, history):
        super().__init__(history)
        self.query_counts = history.groupby('query_norm').size().to_dict()
        matrix = self.micro_matrix.tocsr()
        self.entropy = np.zeros(len(self.texts), dtype=np.float32)
        self.peak = np.zeros(len(self.texts), dtype=np.float32)
        for row in range(len(self.texts)):
            probabilities = matrix.data[matrix.indptr[row]:matrix.indptr[row+1]]
            if len(probabilities):
                self.entropy[row] = -(probabilities*np.log(np.maximum(probabilities,1e-12))).sum()/np.log(max(len(MICROS),2))
                self.peak[row] = probabilities.max()
        self.location_counts = history.groupby('search_location_id').size().to_dict()
        counts = history.groupby(['search_location_id','item_location_id']).size()
        self.location_probability = {}
        for loc, group in counts.groupby(level=0):
            total = float(group.sum())
            self.location_probability[int(loc)] = {int(pair[1]): float(count/total) for pair,count in group.items()}

def v5_features(query, record, history):
    ids, _ = record
    original = rank_semantic_features(query, record)
    row = history.text_to_row.get(query.query_norm)
    known = row is not None
    center = query.search_location_id in centers.index and np.isfinite(centers.loc[query.search_location_id].to_numpy(dtype=float)).all()
    constants = [float(known), np.log1p(history.query_counts.get(query.query_norm,0)),
        float(history.entropy[row]) if known else np.nan,
        float(history.peak[row]) if known else np.nan, float(center)]
    mapping = history.location_probability.get(int(query.search_location_id), {})
    probability = np.array([mapping.get(int(loc),0) for loc in ITEM_LOCS[ids]], dtype=np.float32)
    support = np.full(len(ids),np.log1p(history.location_counts.get(query.search_location_id,0)),dtype=np.float32)
    output = np.column_stack([original,np.tile(constants,(len(ids),1)),probability,support]).astype(np.float32)
    assert output.shape[1] == len(V5_FEATURE_NAMES) and not np.isinf(output).any()
    return output


## 4. Как обучались и проверялись ранкеры

Код ниже сохраняет рецепт обучения: все положительные пары и трудные
кандидаты формируют группы для LightGBM LambdaRank. Обычный запуск notebook
**не переобучает** модели: он использует локальные веса и векторы с
проверкой хешей. Это делает отправленный CSV воспроизводимым на CPU.

In [19]:
def sample_v5_group(ids, base, features, truth, random, miner=None):
    target = np.isin(ids,list(truth))
    positive, negative = np.flatnonzero(target), np.flatnonzero(~target)
    if not len(positive) or not len(negative):
        return np.array([],dtype=np.int32), target
    pieces = [negative[stable_topk(score_candidates(base[negative],best_config),V5_CONFIG['hard_lexical'])],
        negative[stable_topk(base[negative,7],V5_CONFIG['hard_geo_semantic'])],
        negative[stable_topk(base[negative,6],V5_CONFIG['hard_semantic'])]]
    if miner is not None:
        scores = miner.predict(features[negative],num_iteration=200)
        pieces.append(negative[stable_topk(scores,V5_CONFIG['mined_hard'])])
    hard = np.unique(np.concatenate(pieces))
    tail = np.setdiff1d(negative,hard)
    sampled = random.choice(tail,min(len(tail),V5_CONFIG['random_tail']),replace=False)
    selected = np.sort(np.concatenate([positive,hard,sampled])).astype(np.int32)
    assert not set(ids[selected[~target[selected]]]) & truth
    return selected,target

def prepare_v5_training(qframe, source_history, stage, miner=None):
    """Reuse full-pool chunks for mining; never inject a missing positive item."""
    tag = 'mined' if miner is not None else 'initial'
    output_path = CACHE/f'{stage}_{tag}_{fingerprint}.joblib'
    if USE_CACHE and output_path.exists():
        return joblib.load(output_path)
    truths = labels_from_gold(gold,qframe)
    xs,ys,sizes,known_flags,modes,query_positions,audits = [],[],[],[],[],[],[]
    random = np.random.default_rng(SEED+811)
    for mode in ['unseen_text','held_context']:
        assignments = oof_assignments(qframe,mode,3,SEED+803)
        for fold in range(3):
            positions = np.flatnonzero(assignments==fold)
            fold_queries = qframe.iloc[positions]
            fold_truths = [truths[i] for i in positions]
            fit,cold,actual_known = history_for_queries(source_history,fold_queries,fold_truths,
                ITEM_IDS,mode,.9,SEED+804+fold)
            audit = history_coverage(fold_queries,fold_truths,fit,ITEM_IDS)
            audit.update(mode=mode,fold=fold,cold_items=len(cold))
            assert audit['own_context_overlap']==0
            audits.append(audit)
            history = None
            print(stage,tag,mode,'fold',fold,'contexts',len(positions),flush=True)
            for start in range(0,len(positions),V5_CONFIG['chunk']):
                path = CACHE/f'{stage}_pool_{mode}_{fold}_{start}_{fingerprint}.joblib'
                chunk = fold_queries.iloc[start:start+V5_CONFIG['chunk']]
                if USE_CACHE and path.exists():
                    records,matrices = joblib.load(path)
                else:
                    if history is None:
                        history = V5History(fit)
                    records = retrieve_semantic_features(chunk,history,progress_every=100000)
                    matrices = [v5_features(q,r,history) for q,r in zip(chunk.itertuples(index=False),records)]
                    save_cache((records,matrices),path)
                for offset,((ids,base),features) in enumerate(zip(records,matrices)):
                    selected,target = sample_v5_group(ids,base,features,fold_truths[start+offset],random,miner)
                    if not len(selected):
                        continue
                    xs.append(features[selected]); ys.append(target[selected].astype(np.uint8))
                    sizes.append(len(selected)); known_flags.append(bool(actual_known[start+offset]))
                    modes.append(mode); query_positions.append(int(positions[start+offset]))
                if start%1024==0:
                    print(stage,mode,fold,'processed',start+len(chunk),flush=True)
            del history,fit,records,matrices
            gc.collect()
    known = np.array(known_flags,dtype=bool)
    observed = float(known.mean())
    group_weights = np.where(known,known_target/max(observed,1e-6),
        (1-known_target)/max(1-observed,1e-6)).astype(np.float32)
    data = {'X':np.concatenate(xs),'y':np.concatenate(ys),'sizes':np.array(sizes,dtype=np.int32),
        'known':known,'mode':np.array(modes),'query_position':np.array(query_positions,dtype=np.int32),
        'group_weight':group_weights,'audit':audits}
    save_cache(data,output_path)
    report = {'contexts':len(qframe),'groups':len(sizes),'rows':len(data['y']),
        'positives':int(data['y'].sum()),'mean_group_size':float(np.mean(sizes)),
        'actual_known_fraction':observed,'target_known_fraction':known_target,'oof_audits':audits}
    (CACHE/f'{stage}_{tag}_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
    print('Prepared',json.dumps({k:v for k,v in report.items() if k!='oof_audits'}),flush=True)
    return data


### Признаки, веса групп и LightGBM

Положительные пары имеют больший вес; отрицательные выбираются из
лексического, семантического и географического пулов. Отбор модели проведён
по Recall@50 на новых текстах и знакомых текстах с новым контекстом.

In [20]:
def fit_v5(data,name,mode=None,trees=400):
    path = CACHE/f'{name}_{fingerprint}.joblib'
    if USE_CACHE and path.exists():
        return joblib.load(path)
    group_mask = np.ones(len(data['sizes']),dtype=bool) if mode is None else data['mode']==mode
    row_mask = np.repeat(group_mask,data['sizes'])
    x,y = data['X'][row_mask],data['y'][row_mask]
    sizes = data['sizes'][group_mask]
    weights = np.repeat(data['group_weight'][group_mask],sizes)
    if mode=='unseen_text':
        weights[:] = 1
    model = lgb.LGBMRanker(n_estimators=trees,num_leaves=31,learning_rate=.05,
        max_bin=127,min_child_samples=50,reg_lambda=10,random_state=SEED,n_jobs=8,
        verbosity=-1,deterministic=True,force_col_wise=True,
        lambdarank_truncation_level=55,label_gain=[0,1])
    started = time.perf_counter()
    print('Fit',name,'rows',len(y),'groups',len(sizes),flush=True)
    model.fit(x,y,group=sizes,sample_weight=weights)
    save_cache(model,path)
    print('Fitted',name,'seconds',round(time.perf_counter()-started,1),flush=True)
    return model

def evaluation_features(frame,mode,stage,other_frame):
    path = CACHE/f'{stage}_{mode}_{fingerprint}.joblib'
    if USE_CACHE and path.exists():
        return joblib.load(path)
    truths = labels_from_gold(gold,frame)
    other_truths = labels_from_gold(gold,other_frame)
    other_positive = np.array(sorted({ITEM_IDS[i] for truth in other_truths for i in truth}),dtype=str)
    random = np.random.default_rng(SEED+812)
    other_cold = set(random.choice(other_positive,int(.9*len(other_positive)),replace=False))
    fit,cold,known = history_for_queries(history_all,frame,truths,ITEM_IDS,mode,.9,SEED+801,
        excluded_texts=other_frame.query_norm,excluded_items=other_cold)
    history = V5History(fit)
    records = retrieve_semantic_features(frame,history,progress_every=500)
    matrices = [v5_features(q,r,history) for q,r in zip(frame.itertuples(index=False),records)]
    output = (records,matrices,known,history_coverage(frame,truths,fit,ITEM_IDS))
    save_cache(output,path)
    del history,fit
    gc.collect()
    return output

def predict_scores(model,matrices,trees=200):
    output=[]
    for start in range(0,len(matrices),64):
        block=matrices[start:start+64]
        values=model.predict(np.concatenate(block),num_iteration=trees)
        output.extend(np.split(values,np.cumsum([len(x) for x in block])[:-1]))
    return output

def top50(records,scores):
    return [ids[stable_topk(score,50)] for (ids,_),score in zip(records,scores)]

def old_v4_scores(records,matrices,final=False):
    """Frozen external comparator; never used as a new training feature."""
    priors=ROOT/'models/retrieval-priors'
    prefix='final' if final else 'evaluation'
    legacy=joblib.load(priors/f'{prefix}_legacy.joblib')
    semantic=joblib.load(priors/f'{prefix}_semantic.joblib')
    model_path=OLD_CACHE/('final_ranker_c0442eeba2d244f2.joblib' if final else 'lgb_rank_expanded_200_c0442eeba2d244f2.joblib')
    ranker=joblib.load(model_path)
    a=model_scores(legacy,[x[:,:32] for x in matrices])
    b=model_scores(semantic,[x[:,:38] for x in matrices])
    c=predict_scores(ranker,[x[:,:38] for x in matrices],200)
    result=[]
    for (_,base),la,se,ra in zip(records,a,b,c):
        lexical=score_candidates(base,best_config)
        first=.5/(60+rankdata(-la,method='min'))+.5/(60+rankdata(-lexical,method='min'))
        prior=.25/(60+rankdata(-se,method='min'))+.75/(60+rankdata(-first,method='min'))
        result.append(.75/(60+rankdata(-ra,method='min'))+.25/(60+rankdata(-prior,method='min')))
    return result


### Слияние оценок и локальная метрика

Ранги из нескольких каналов объединяются reciprocal rank fusion (RRF).
Предел в 50 объявлений применяется после объединения всех сигналов.

In [21]:
def blend_scores(scores,baseline,weight):
    if weight==0:
        return baseline
    if weight==1:
        return scores
    return [weight/(60+rankdata(-s,method='min'))+(1-weight)/(60+rankdata(-b,method='min'))
            for s,b in zip(scores,baseline)]

def matched_slice_recall(frame,recalls,target,mask=None):
    """Match query-length/filter/coordinate-presence strata within the actual slice."""
    from collections import Counter
    def strata(f):
        return list(zip(np.digitize(f.query_norm.str.len().to_numpy(),[15,25]).tolist(),
            (f.search_infm_params_text.str.len()>0).tolist(),f.search_location_id.isin(centers.index).tolist()))
    if mask is None:
        mask=np.ones(len(frame),dtype=bool)
    actual=strata(frame.loc[mask]); observed=Counter(actual); desired=Counter(strata(target))
    missing=set(desired)-set(observed)
    if missing:
        raise ValueError(f'Unsupported evaluation strata: {missing}')
    weights=np.array([desired.get(k,0)/observed[k] for k in actual],dtype=float)
    weights/=weights.sum()
    return float(recalls[mask]@weights)

def development_metrics(unseen_prediction,held_prediction,held_known):
    truths=labels_from_gold(gold,development)
    a=per_query_recall(unseen_prediction,truths)
    b=per_query_recall(held_prediction,truths)
    known=queries.query_norm.isin(history_all.query_norm)
    unknown_score=matched_slice_recall(development,a,queries[~known])
    known_score=matched_slice_recall(development,b,queries[known],held_known)
    return {'matched_recall50':(1-known_target)*unknown_score+known_target*known_score,
        'unseen_macro_recall50':float(a.mean()),'actual_known_macro_recall50':float(b[held_known].mean()),
        'held_context_macro_recall50':float(b.mean()),'held_actual_known_fraction':float(held_known.mean())}


### Рецепт обучения и экспорт весов

Функции оставлены в notebook, чтобы проверяющий видел путь обучения.
Выполняемый в конце экспорт обращается к сохранённым финальным весам.

In [22]:
def main():
    initial=prepare_v5_training(training,training_history,'evaluation')
    cold_model=fit_v5(initial,'cold_wide',mode='unseen_text')
    mixed_model=fit_v5(initial,'mixed_wide')
    del initial
    gc.collect()
    mined=prepare_v5_training(training,training_history,'evaluation',miner=mixed_model)
    mined_model=fit_v5(mined,'mixed_mined')
    del mined
    gc.collect()
    unseen_records,unseen_features,_,unseen_audit=evaluation_features(development,'unseen_text','development',control)
    held_records,held_features,held_known,held_audit=evaluation_features(development,'held_context','development',control)
    unseen_baseline=old_v4_scores(unseen_records,unseen_features)
    held_baseline=old_v4_scores(held_records,held_features)
    baseline=development_metrics(top50(unseen_records,unseen_baseline),top50(held_records,held_baseline),held_known)
    rows=[{'model':'v4','trees':200,'weight':0.,**baseline}]
    models={'cold_wide':cold_model,'mixed_wide':mixed_model,'mixed_mined':mined_model}
    for name,model in models.items():
        for trees in [100,200,400]:
            a=predict_scores(model,unseen_features,trees)
            b=predict_scores(model,held_features,trees)
            for weight in [.5,.75,1.]:
                metrics=development_metrics(top50(unseen_records,blend_scores(a,unseen_baseline,weight)),
                    top50(held_records,blend_scores(b,held_baseline,weight)),held_known)
                rows.append({'model':name,'trees':trees,'weight':weight,**metrics})
            print('Development',name,trees,'best',max(r['matched_recall50'] for r in rows if r['model']==name),flush=True)
    table=pd.DataFrame(rows).sort_values(['matched_recall50','model','trees','weight'],ascending=[False,True,True,True])
    table.to_csv(CACHE/'development_experiments.csv',index=False)
    admissible=table[table.unseen_macro_recall50>=baseline['unseen_macro_recall50']-.005]
    winner=admissible.iloc[0].to_dict()
    selection={'winner':winner,'baseline':baseline,'unseen_history':unseen_audit,'held_history':held_audit,
        'fingerprint':fingerprint,'config':V5_CONFIG,'feature_names':V5_FEATURE_NAMES,
        'control_used_for_selection':False,'frozen_comparator_limitation':'v4 auxiliary history may have used held control interactions'}
    (CACHE/'selection.json').write_text(json.dumps(selection,indent=2),encoding='utf-8')
    print('Selected on development',json.dumps(winner),flush=True)
    del unseen_records,unseen_features,held_records,held_features
    gc.collect()
    # Only now open the held control. It never changes the selected settings.
    records,features,_,audit=evaluation_features(control,'unseen_text','control',development)
    old_scores=old_v4_scores(records,features)
    if winner['model']=='v4':
        selected_scores=old_scores
    else:
        selected_scores=blend_scores(predict_scores(models[winner['model']],features,int(winner['trees'])),old_scores,float(winner['weight']))
    truth=labels_from_gold(gold,control)
    a=per_query_recall(top50(records,old_scores),truth)
    b=per_query_recall(top50(records,selected_scores),truth)
    random=np.random.default_rng(SEED+813)
    delta=b-a
    bootstrap=[float(random.choice(delta,len(delta),replace=True).mean()) for _ in range(4000)]
    result={'contexts':len(control),'v4_recall50':float(a.mean()),'selected_recall50':float(b.mean()),
        'delta':float(delta.mean()),'paired_delta_bootstrap95':np.quantile(bootstrap,[.025,.975]).tolist(),
        'improved':int((delta>0).sum()),'worsened':int((delta<0).sum()),
        'pool_recall':float(per_query_recall([r[0] for r in records],truth).mean()),'history_audit':audit,
        'used_for_selection':False,'frozen_comparator_limitation':selection['frozen_comparator_limitation']}
    (CACHE/'control_metrics.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
    print('CONTROL',json.dumps(result),flush=True)
    final_fit_and_export(winner,models['mixed_wide'],selection,result)

def final_fit_and_export(winner,miner,selection,control_result):
    if winner['model']=='v4':
        print('No validated development improvement; kept original answer.csv.',flush=True)
        return
    frame=history_all[history_all.context_key.isin(gold)].drop_duplicates('context_key')
    frame=frame[[*QUERY_COLS,'query_norm','context_key']].sort_values('context_key').reset_index(drop=True)
    final_path=CACHE/f'final_ranker_{fingerprint}.joblib'
    if USE_CACHE and final_path.exists():
        model=joblib.load(final_path)
    else:
        data=prepare_v5_training(frame,history_all,'final',miner=miner if winner['model']=='mixed_mined' else None)
        model=fit_v5(data,'final_ranker',mode='unseen_text' if winner['model']=='cold_wide' else None,trees=int(winner['trees']))
        del data
        gc.collect()
    history=V5History(history_all)
    path=CACHE/f'benchmark_{fingerprint}.joblib'
    if USE_CACHE and path.exists():
        records,features=joblib.load(path)
    else:
        records=retrieve_semantic_features(queries,history,progress_every=500)
        features=[v5_features(q,r,history) for q,r in zip(queries.itertuples(index=False),records)]
        save_cache((records,features),path)
    score=predict_scores(model,features,int(winner['trees']))
    if float(winner['weight'])<1:
        score=blend_scores(score,old_v4_scores(records,features,final=True),float(winner['weight']))
    prediction=top50(records,score)
    answer=pd.DataFrame({'query_id':queries.query_id.astype(str),
        'answer':[' '.join(ITEM_IDS[ids]) for ids in prediction]})
    assert list(answer.columns)==['query_id','answer'] and len(answer)==len(queries)
    assert answer.query_id.is_unique and set(answer.query_id)==set(queries.query_id)
    valid=set(ITEM_IDS)
    for text in answer.answer:
        values=text.split(' ')
        assert len(values)==50 and len(set(values))==50 and all(value in valid and re.fullmatch('[0-9a-f]{16}',value) for value in values)
    # Keep an explicit byte-identical copy of the previously submitted version.
    previous=ROOT/'answer_v0.4.csv'
    if not previous.exists():
        previous.write_bytes((ROOT/'answer.csv').read_bytes())
    answer.to_csv(CACHE/'answer_candidate.csv',index=False,encoding='utf-8',lineterminator='\n')
    loaded=pd.read_csv(CACHE/'answer_candidate.csv',dtype=str,keep_default_na=False)
    assert loaded.equals(answer)
    repeated=top50(records,blend_scores(predict_scores(model,features,int(winner['trees'])),
        old_v4_scores(records,features,final=True),float(winner['weight'])) if float(winner['weight'])<1 else predict_scores(model,features,int(winner['trees'])))
    assert all(np.array_equal(a,b) for a,b in zip(prediction,repeated))
    (ROOT/'answer.csv').write_bytes((CACHE/'answer_candidate.csv').read_bytes())
    manifest={'stage':'ranking-v5','fingerprint':fingerprint,'input_sha256':input_hashes,
        'pipeline_sha256':'f92b61f6e92cd3893c8043e95edabd7f9bb6985998e31024107226be193b400b',
        'selection':selection,'control':control_result,'final_contexts':len(frame),
        'feature_names':V5_FEATURE_NAMES,'answer_sha256':sha256_file(ROOT/'answer.csv'),
        'previous_answer_sha256':sha256_file(previous),'final_model':f'final_ranker_{fingerprint}.joblib',
        'versions':{'python':sys.version.split()[0],'lightgbm':lgb.__version__,'numpy':np.__version__,
                    'pandas':pd.__version__,'sklearn':sklearn.__version__}}
    (CACHE/'manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    print('FINAL',json.dumps({'answer_sha256':manifest['answer_sha256'],'contexts':len(frame),'winner':winner}),flush=True)


In [23]:
# Settings frozen before the held control was opened.
FROZEN_SELECTION = {'winner': {'model': 'mixed_mined', 'trees': 400, 'weight': 0.75, 'matched_recall50': 0.9476002382121281, 'unseen_macro_recall50': 0.9434558823529412, 'actual_known_macro_recall50': 0.9504106776180699, 'held_context_macro_recall50': 0.9441911764705883, 'held_actual_known_fraction': 0.7161764705882353}, 'baseline': {'matched_recall50': 0.9456678023220599, 'unseen_macro_recall50': 0.9422794117647059, 'actual_known_macro_recall50': 0.9497946611909651, 'held_context_macro_recall50': 0.9416911764705882, 'held_actual_known_fraction': 0.7161764705882353}, 'unseen_history': {'contexts': 3400, 'positive_pairs': 3480, 'known_text_fraction': 0.0, 'positive_item_seen_macro_fraction': 0.03200980392156862, 'own_context_overlap': 0}, 'held_history': {'contexts': 3400, 'positive_pairs': 3480, 'known_text_fraction': 0.7161764705882353, 'positive_item_seen_macro_fraction': 0.036274509803921565, 'own_context_overlap': 0}, 'fingerprint': 'e2854ae5a86a1bbf', 'config': {'data_revision': 1, 'chunk': 128, 'folds': 3, 'cold_fraction': 0.9, 'hard_lexical': 100, 'hard_geo_semantic': 100, 'hard_semantic': 60, 'random_tail': 64, 'mined_hard': 160, 'trees': 400}, 'feature_names': ['title_bm25', 'body_bm25', 'char_tfidf', 'filter_bm25', 'geo_compatibility', 'micro_compatibility', 'baseline_score', 'log_baseline_rank', 'log_title_rank', 'log_body_rank', 'log_char_rank', 'same_location', 'log_distance_km', 'title_token_coverage', 'body_token_coverage', 'title_idf_coverage', 'body_idf_coverage', 'query_in_title', 'query_equals_title', 'query_characters', 'query_tokens', 'log_filter_characters', 'rating_required', 'rating_gap', 'rating', 'log_reviews', 'log_price', 'phone_hidden', 'messages_forbidden', 'log_title_characters', 'log_description_characters', 'log_params_characters', 'e5_cosine', 'e5_geo_affinity', 'from_lexical_pool', 'from_semantic_pool', 'log_e5_rank', 'log_e5_geo_rank', 'history_query_known', 'history_query_log_pairs', 'history_query_micro_entropy', 'history_query_top_micro_probability', 'geo_center_available', 'geo_transition_probability', 'geo_transition_log_support'], 'control_used_for_selection': False, 'frozen_comparator_limitation': 'v4 auxiliary history may have used held control interactions'}
FROZEN_CONTROL = {'contexts': 600, 'v4_recall50': 0.94, 'selected_recall50': 0.9466666666666667, 'delta': 0.006666666666666667, 'paired_delta_bootstrap95': [-0.0033333333333333335, 0.018333333333333333], 'improved': 7, 'worsened': 3, 'pool_recall': 0.995, 'history_audit': {'contexts': 600, 'positive_pairs': 604, 'known_text_fraction': 0.0, 'positive_item_seen_macro_fraction': 0.0016666666666666668, 'own_context_overlap': 0}, 'used_for_selection': False, 'frozen_comparator_limitation': 'v4 auxiliary history may have used held control interactions'}


## 5. Зафиксированные компоненты итогового каскада

Большие конфигурационные словари вынесены в `config/solution_v11.json` для
удобства чтения. Хеш конфигурации проверяется до применения. Все модели,
векторы и корпусные справочники читаются локально; существенные файлы также
проверяются по SHA-256 перед использованием.

### Манифест поиска и проверка файлов

В конфигурации записаны пути, версии, параметры модели и хеши артефактов.
Загрузка не зависит от конкретного `query_id` и не содержит тестовой разметки.

In [24]:
SOLUTION_CONFIG_PATH = ROOT / 'config/solution_v12.json'
assert sha256_file(SOLUTION_CONFIG_PATH) == 'f17b0668c3da8e6d13702f51d8552eda2502fc2c4dd734e1ca027385036ac449'
SOLUTION_CONFIG = json.loads(SOLUTION_CONFIG_PATH.read_text(encoding='utf-8'))
FROZEN_QUERY_MANIFEST = SOLUTION_CONFIG['retrieval']
V5_CACHE = CACHE
V5_FP = fingerprint
V5_MANIFEST = json.loads((V5_CACHE/'manifest.json').read_text(encoding='utf-8'))
V5_WINNER = V5_MANIFEST['selection']['winner']
assert FROZEN_QUERY_MANIFEST['input_sha256'] == input_hashes
QE_CONFIG = {'max_query_length':64}
encoder_path = ROOT/FROZEN_QUERY_MANIFEST['encoder']
assert sha256_file(encoder_path/'model.safetensors') == FROZEN_QUERY_MANIFEST['encoder_sha256']


In [25]:
def baseline_v5(records, matrices, final=False):
    name = V5_MANIFEST['final_model'] if final else f'mixed_mined_{V5_FP}.joblib'
    ranker = joblib.load(V5_CACHE/name)
    return blend_scores(predict_scores(ranker, matrices, int(V5_WINNER['trees'])),
                        old_v4_scores(records, matrices, final=final), float(V5_WINNER['weight']))


### Кодирование запросов

Функция ниже нужна, если готовые векторы запросов не найдены. Она работает
локально и использует те же замороженные веса.

In [26]:
def encode_learned(model,tokenizer,texts):
    order=np.argsort([len(text) for text in texts],kind='stable')
    output=np.empty((len(texts),384),np.float32)
    model.eval()
    with torch.inference_mode():
        for start in range(0,len(order),128):
            positions=order[start:start+128]
            batch=tokenizer(['query: '+texts[i] for i in positions],padding=True,truncation=True,
                max_length=QE_CONFIG['max_query_length'],return_tensors='pt')
            batch={key:value.to(DEVICE) for key,value in batch.items()}
            hidden=model(**batch).last_hidden_state
            mask=batch['attention_mask'].unsqueeze(-1)
            pooled=(hidden*mask).sum(dim=1)/mask.sum(dim=1).clamp(min=1)
            output[positions]=torch.nn.functional.normalize(pooled,p=2,dim=1).cpu().numpy()
    assert np.isfinite(output).all() and np.allclose(np.linalg.norm(output,axis=1),1,atol=2e-5)
    return output


In [27]:
def query_features(vector,ids,base):
    cosine=(semantic_index.items[ids].astype(np.float64)@vector.astype(np.float64)).astype(np.float32)
    geographic=semantic_affinity(cosine,base[:,4])
    return np.column_stack([cosine,geographic,cosine-base[:,6],geographic-base[:,7],
        np.log1p(rankdata(-cosine,method='min')),np.log1p(rankdata(-geographic,method='min'))]).astype(np.float32)


### Подбор соседних запросов и контекстное расширение

Соседи дают дополнительную вероятность подкатегории, но итоговый выбор
остаётся за ранкером, который видит также прямые признаки объявления.

In [28]:
LEARNED_LOOKUP = {}
GEO_EVIDENCE = None

def retrieve_context_features(query_frame, history_model, progress_every=200):
    texts = query_frame.query_norm.astype(str).tolist()
    title_queries = lexical["title"].query_matrix(texts)
    body_queries = lexical["body"].query_matrix(texts)
    filter_queries = lexical["body"].query_matrix(query_frame.search_infm_params_text.astype(str).tolist())
    char_queries = lexical["char_vectorizer"].transform(texts)
    micro_predictions = history_model.micro_predictions(texts)
    outputs, geo_cache = [], {}
    started = time.perf_counter()
    dense_block = None
    for i, query in enumerate(query_frame.itertuples(index=False)):
        if i % 64 == 0:
            dense_block = semantic_index.scores(texts[i:i + 64])
        cosine = dense_block[i % 64]
        if i % 64 == 0:
            learned_vectors = np.stack([LEARNED_LOOKUP[text] for text in texts[i:i+64]])
            learned_block = (learned_vectors.astype(np.float64) @ semantic_index.item_transpose).astype(np.float32)
        learned_cosine = learned_block[i % 64]
        title = normalized_scores((title_queries[i] @ lexical["title"].transpose).toarray().ravel())
        body = normalized_scores((body_queries[i] @ lexical["body"].transpose).toarray().ravel())
        chars = normalized_scores((char_queries[i] @ lexical["char_transpose"]).toarray().ravel())
        filters = normalized_scores((filter_queries[i] @ lexical["body"].transpose).toarray().ravel())
        loc = int(query.search_location_id)
        if loc not in geo_cache:
            geo_cache[loc] = history_model.geography(loc)
        geo = geo_cache[loc]
        micro = micro_predictions[i][ITEM_MICRO_COLS]
        candidate_parts = []
        for source in [title, body, chars]:
            candidate_parts.append(stable_topk(source, CONFIG["retrieval_pool"]))
            candidate_parts.append(stable_topk(source * (0.03 + 0.97 * geo), CONFIG["retrieval_pool"]))
        mixed = 0.5 * title + 0.3 * body + 0.2 * chars
        candidate_parts.append(stable_topk(mixed * (0.03 + 0.97 * geo) * (0.2 + 0.8 * micro), CONFIG["retrieval_pool"]))
        lexical_candidates = np.unique(np.concatenate(candidate_parts))
        affinity = semantic_affinity(cosine, geo)
        semantic_candidates = np.union1d(stable_topk(cosine, SEMANTIC_CONFIG["retrieval_pool"]),
                                          stable_topk(affinity, SEMANTIC_CONFIG["retrieval_pool"]))
        learned_candidates = np.union1d(stable_topk(learned_cosine, 500),
                                          stable_topk(semantic_affinity(learned_cosine, geo), 500))
        extra_geo = GEO_EVIDENCE.additional_candidates(loc, cosine, learned_cosine,
            items.item_latitude.to_numpy(), items.item_longitude.to_numpy(), stable_topk)
        semantic_candidates = np.union1d(semantic_candidates, np.union1d(learned_candidates, extra_geo))
        candidates = np.union1d(lexical_candidates, semantic_candidates)
        features = np.column_stack([title[candidates], body[candidates], chars[candidates],
            filters[candidates], geo[candidates], micro[candidates], cosine[candidates], affinity[candidates],
            np.isin(candidates, lexical_candidates), np.isin(candidates, semantic_candidates)]).astype(np.float32)
        outputs.append((candidates, features))
        if (i + 1) % progress_every == 0 or i + 1 == len(query_frame):
            print(f"Hybrid retrieved {i + 1}/{len(query_frame)}; {time.perf_counter() - started:.1f}s", flush=True)
    return outputs


### Контекстные признаки и обученный query encoder

Обученный query encoder E5 сопоставляет запрос с объявлением, а история и
соседние запросы добавляют сигналы подкатегории и локации. Документный
encoder остаётся замороженным, поэтому векторы корпуса можно переиспользовать.

### Базовый ранкер v10

Базовая модель учитывает лексические и E5-признаки, историю, категорию,
географию, поля объявления и BGE-M3. Она служит также вторым членом
финальной смеси: новый ранкер не заменяет её полностью.

In [29]:
QUALITY_V10_MANIFEST = SOLUTION_CONFIG['quality_v10']


### Добавленные признаки v11

Десять новых признаков проверяют явный вид и тип услуги, распространённость
объявлений в локации, возможность выезда или удалённой работы и их связь с
расстоянием. Они улучшают отбор без жёсткого удаления кандидатов. Новый
LightGBM обучен на 39,67 млн пар; его ранг смешивается с рангом v10 50/50.

In [30]:
QUALITY_V11_MANIFEST = SOLUTION_CONFIG['quality_v12']


## 6. Генерация и проверка `answer.csv`

Эта функция загружает сохранённые векторы и модели, строит полный пул,
считает признаки v10 и v11, объединяет ранги и сохраняет до 50 строковых
`item_id` на запрос. В конце проверяются количество строк, уникальность,
формат и принадлежность корпусу, затем SHA-256 всего CSV.

In [31]:
def export_frozen_query_candidate():
    global LEARNED_LOOKUP, GEO_EVIDENCE
    # 1. Запросные векторы: готовый локальный кэш или локальный encoder.
    vector_path=ROOT/FROZEN_QUERY_MANIFEST['query_vectors']
    if USE_CACHE and vector_path.exists():
        assert sha256_file(vector_path)==FROZEN_QUERY_MANIFEST['query_vectors_sha256']
        vectors=np.load(vector_path,allow_pickle=False)
    else:
        model=AutoModel.from_pretrained(encoder_path,local_files_only=True,attn_implementation='eager').to(DEVICE)
        tokenizer=AutoTokenizer.from_pretrained(MODEL_DIR,local_files_only=True)
        vectors=encode_learned(model,tokenizer,sorted(set(queries.query_norm)))
        del model,tokenizer
        gc.collect()
        if DEVICE=='cuda':torch.cuda.empty_cache()
    assert vectors.shape==(queries.query_norm.nunique(),384) and np.isfinite(vectors).all()
    lookup={text:i for i,text in enumerate(sorted(set(queries.query_norm)))}
    LEARNED_LOOKUP = {text:vectors[row] for text,row in lookup.items()}
    # 2. Исторические агрегаты обучения и географические соседи.
    # The frozen bank contains training aggregates, not benchmark answers.
    import sys
    sys.path.insert(0,str(ROOT/'.development'))
    from context_signals_v9 import SemanticEvidence
    assert sha256_file(ROOT/'.development/context_signals_v9.py')==FROZEN_QUERY_MANIFEST['context_module_sha256']
    bank_path=ROOT/FROZEN_QUERY_MANIFEST['history_bank']
    assert sha256_file(bank_path)==FROZEN_QUERY_MANIFEST['history_bank_sha256']
    bank=joblib.load(bank_path);GEO_EVIDENCE=bank['geo']
    evidence_path=ROOT/FROZEN_QUERY_MANIFEST['query_evidence']
    assert sha256_file(evidence_path)==FROZEN_QUERY_MANIFEST['query_evidence_sha256']
    evidence=joblib.load(evidence_path)
    neighbor_lookup={text:i for i,text in enumerate(evidence['texts'])}
    neighbor_results=tuple(array[[neighbor_lookup[text] for text in queries.query_norm]] for array in evidence['results'])
    # 3. Широкий лексико-семантический пул и базовые признаки.
    pool_path=V5_CACHE/f'expanded_benchmark_{FROZEN_QUERY_MANIFEST["fingerprint"]}.joblib'
    if USE_CACHE and pool_path.exists():
        records,features=joblib.load(pool_path)
    else:
        history=V5History(history_all)
        records=retrieve_context_features(queries,history,progress_every=500)
        features=[v5_features(query,record,history) for query,record in zip(queries.itertuples(index=False),records)]
        del history
    # 4. Базовый скоринг с обученным query encoder.
    original=baseline_v5(records,features,final=True)
    added=[query_features(vectors[lookup[text]],ids,base) for text,(ids,base) in zip(queries.query_norm,records)]
    winner=FROZEN_QUERY_MANIFEST['selection']['winner']
    if winner['variant']=='direct_query_blend':
        score=blend_scores([x[:,1] for x in added],original,float(winner['weight']))
    else:
        path=ROOT/FROZEN_QUERY_MANIFEST['final_ranker']
        assert sha256_file(path)==FROZEN_QUERY_MANIFEST['final_ranker_sha256']
        model=joblib.load(path)
        assert model.n_features_in_==51
        score=blend_scores(predict_scores(model,[np.column_stack([x,y]) for x,y in zip(features,added)],int(winner['trees'])),original,float(winner['weight']))
    # 5. Контекстные, географические и соседские признаки.
    context_matrices=[]
    for row,(query,(ids,raw),base,learned) in enumerate(zip(queries.itertuples(index=False),records,features,added)):
        geography=GEO_EVIDENCE.features(query.search_location_id,items.item_latitude.to_numpy()[ids],items.item_longitude.to_numpy()[ids])
        neighborhood=SemanticEvidence.features(neighbor_results[0][row],neighbor_results[1][row],neighbor_results[2][row],
            semantic_index.items[ids],ITEM_MICRO_COLS[ids],rankdata)
        context_matrices.append(np.column_stack([base,learned,geography,neighborhood]).astype(np.float32))
    context_path=FROZEN_QUERY_MANIFEST['context_ranker']
    if context_path:
        assert sha256_file(ROOT/context_path)==FROZEN_QUERY_MANIFEST['context_ranker_sha256']
        context_model=joblib.load(ROOT/context_path)
        columns=FROZEN_QUERY_MANIFEST['context_columns']
        new_scores=predict_scores(context_model,[np.ascontiguousarray(x[:,columns]) for x in context_matrices],400)
        score=blend_scores(new_scores,score,FROZEN_QUERY_MANIFEST['context_selection']['winner']['weight'])
    ce_reference=score
    # 6. Финальный ранкер v10 на текстовых, исторических и BGE-признаках.
    from quality_model_v10 import adjust_quality_scores
    score=adjust_quality_scores(ROOT,QUALITY_V10_MANIFEST,queries,items,records,context_matrices,score,
        semantic_index.items,sha256_file,blend_scores,predict_scores)
    from quality_cross_model_v10 import adjust_cross_scores
    score=adjust_cross_scores(ROOT,QUALITY_V10_MANIFEST,queries,items,records,score,ce_reference,
        sha256_file,blend_scores,stable_topk)
    # 7. Структурный ранкер v11 и слияние рангов 50/50.
    from quality_model_v11 import adjust_v11_scores
    score=adjust_v11_scores(ROOT,QUALITY_V10_MANIFEST,QUALITY_V11_MANIFEST,
        queries,items,records,context_matrices,score,semantic_index.items,
        sha256_file,blend_scores,predict_scores)
    # 8. Выбор top-50 и проверка идентификаторов до сохранения.
    predictions=top50(records,score)
    answer=pd.DataFrame({'query_id':queries.query_id.astype(str),'answer':[' '.join(ITEM_IDS[prediction]) for prediction in predictions]})
    assert list(answer.columns)==['query_id','answer'] and len(answer)==len(queries)==2452
    assert answer.query_id.is_unique and set(answer.query_id)==set(queries.query_id)
    allowed=set(ITEM_IDS)
    for row in answer.itertuples(index=False):
        assert len(row.query_id)==16
        ids=row.answer.split(' ')
        assert len(ids)==50 and len(set(ids))==50
        assert all(re.fullmatch('[0-9a-f]{16}',item) and item in allowed for item in ids)
    # 9. Побайтовая проверка против отправленного файла.
    output=ROOT/FROZEN_QUERY_MANIFEST['answer_file']
    output.parent.mkdir(parents=True,exist_ok=True)
    answer.to_csv(output,index=False,encoding='utf-8',lineterminator='\n')
    if FROZEN_QUERY_MANIFEST['answer_sha256'] is not None:
        assert sha256_file(output)==FROZEN_QUERY_MANIFEST['answer_sha256']
    print('Validated solution:',output,'SHA256:',sha256_file(output))
    return answer

FROZEN_QUERY_MANIFEST['answer_file']=QUALITY_V11_MANIFEST['answer_file']
FROZEN_QUERY_MANIFEST['answer_sha256']=QUALITY_V11_MANIFEST['answer_sha256']
query_answer=export_frozen_query_candidate()
query_answer.head()


Hybrid retrieved 500/2452; 10.7s
Hybrid retrieved 1000/2452; 20.6s
Hybrid retrieved 1500/2452; 30.2s
Hybrid retrieved 2000/2452; 41.3s
Hybrid retrieved 2452/2452; 50.8s
Validated solution: D:\JupyterProjects\AvitoTest\experiments\results\v12\answer.csv SHA256: f4d83da229b9099fbe2e7725158987bf87bc7c1f68ccafb5c7c3c0e7b2d99fe8
